# K3D-11 · Выход через DLPack-слоты: движок v9 (~13 мин GPU)

K3D-10 разложил холодный вызов v7 H16 на T4 (режим официального цикла):
- проверки входа — 6 мкс;
- **`new_empty` (выделение выхода) — 30 мкс**;
- **первый запуск ядра — 19 мкс**;
- дальше `synchronize` харнесса.

Пробуждение GPU и ранний stamp не помогли.

**v9:** выход берётся из пула слотов на GPU и оборачивается в тензор через DLPack (`torch._C._from_dlpack`) — без диспетчера и без кэширующего аллокатора. Указатель слота известен сразу, поэтому ядро запускается до создания тензора: GPU работает, пока CPU собирает объект. Слот занят, пока жив тензор (deleter DLPack), — представления (views) и удержанные выходы его держат. Если свободного слота нет, выход выделяется через `new_empty`, как раньше.

**Варианты** (расширение `k3d_fast9` с ядрами v7, метки rdtsc этапов):
- C-путь без запуска: `new_empty` / DLPack;
- пустое ядро: `new_empty` / DLPack + запуск первым;
- v7 H16: `new_empty` / DLPack / DLPack + запуск первым;
- рабочие версии: v7 H16, v6 H64 (лидерборд) и **зип v9 H16 как есть**.

**Режимы:** подряд, loop, idle. Плюс сверка с эталоном и проверка, что все слоты освобождаются.

**Настройки:** GPU T4 x2, Internet On. **Save Version → Save & Run All (Commit)**. Работает и в Colab.

**После завершения:** `k3d11_bench.zip` из Output — в папку проекта.

In [ ]:
# 0. Конфигурация (порядок устройств как у NVML — до импорта torch)
import os
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
import sys, json, time, shutil, re, io, zipfile, subprocess
from pathlib import Path
import numpy as np
import torch
WORK = Path("/kaggle/working")  # on Colab this folder is simply created (the %%writefile cells below write there)
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
os.environ["K3D_WORK"] = str(WORK)
for p in (WORK / "stub" / "kandi3d_image23d", WORK / "sub6" / "weights", WORK / "sub7" / "weights", WORK / "x9", WORK / "bench", WORK / "k3d" / "calib"):
    p.mkdir(parents=True, exist_ok=True)
print(sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.version.cuda, "|",
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "Нет GPU: включи Accelerator = GPU T4 x2"


In [ ]:
%%writefile /kaggle/working/stub/kandi3d_image23d/runtime.py
"""Runtime / sampling parameters for Image23D inference (separate from architecture config)."""

from __future__ import annotations

from dataclasses import asdict, dataclass, field, replace
from typing import Any, Dict, Optional, Tuple, TypeVar

T = TypeVar("T", bound="RuntimeParams")


@dataclass(frozen=True)
class RuntimeParams:
    """Base runtime params with None-aware merge."""

    def merge(self: T, other: Optional[T] = None, **kwargs: Any) -> T:
        base = asdict(self)
        if other is not None:
            for k, v in asdict(other).items():
                if v is not None:
                    base[k] = v
        for k, v in kwargs.items():
            if v is not None and k in base:
                if k == "cfg_range" and isinstance(v, (list, tuple)):
                    base[k] = tuple(float(x) for x in v)
                else:
                    base[k] = v
        return type(self)(**base)


@dataclass(frozen=True)
class FlowMatchSamplingParams(RuntimeParams):
    guidance_scale: float = 2.5
    cfg_range: Tuple[float, float] = (0.5, 1.0)
    rescale_t: float = 3.0
    num_inference_steps: int = 50

    @classmethod
    def from_yaml_section(cls, section: Dict[str, Any]) -> "FlowMatchSamplingParams":
        cfg_range = section.get("cfg_range", [0.5, 1.0])
        if isinstance(cfg_range, (list, tuple)):
            cfg_range = tuple(float(x) for x in cfg_range)
        else:
            cfg_range = (0.5, 1.0)
        return cls(
            guidance_scale=float(section.get("cfg", section.get("guidance_scale", 2.5))),
            cfg_range=cfg_range,
            rescale_t=float(section.get("rescale_t", 3.0)),
            num_inference_steps=int(section.get("num_inference_steps", 50)),
        )

    @classmethod
    def from_dict(cls, data: Optional[Dict[str, Any]]) -> "FlowMatchSamplingParams":
        if not data:
            return cls()
        cfg_range = data.get("cfg_range", (0.5, 1.0))
        if isinstance(cfg_range, list):
            cfg_range = tuple(cfg_range)
        return cls(
            guidance_scale=float(data.get("guidance_scale", 2.5)),
            cfg_range=tuple(cfg_range),
            rescale_t=float(data.get("rescale_t", 3.0)),
            num_inference_steps=int(data.get("num_inference_steps", 50)),
        )


@dataclass(frozen=True)
class PcdVaeDecodeParams(RuntimeParams):
    """Placeholder for future PcdVae decode-time runtime parameters."""

    @classmethod
    def from_geo_config(cls, geo_config: Dict[str, Any]) -> "PcdVaeDecodeParams":
        return cls()

    @classmethod
    def from_dict(cls, data: Optional[Dict[str, Any]]) -> "PcdVaeDecodeParams":
        return cls()


@dataclass(frozen=True)
class PcdRepGeometryParams(RuntimeParams):
    max_chunk: Optional[int] = None
    hdecode_levels: int = 3

    @classmethod
    def from_geo_config(cls, geo_config: Dict[str, Any]) -> "PcdRepGeometryParams":
        gen = geo_config.get("generators", {}).get("pcd_vae", {})
        return cls(max_chunk=int(gen.get("max_rep_length", 262144)))

    @classmethod
    def from_dict(cls, data: Optional[Dict[str, Any]]) -> "PcdRepGeometryParams":
        if not data:
            return cls()
        return cls(
            max_chunk=data.get("max_chunk"),
            hdecode_levels=int(data.get("hdecode_levels", 3)),
        )


@dataclass(frozen=True)
class LatDitConditioning:
    """Image conditioning bundle for LatDit (replaces data_dict cond keys)."""

    dino_features: "torch.Tensor"
    dino_token: Optional["torch.Tensor"] = None
    batch_size: int = 1
    nfaces: Optional["torch.Tensor"] = None

    def to_data_dict(self) -> Dict[str, Any]:
        dd: Dict[str, Any] = {
            "dino_features": self.dino_features,
            "batch_size": self.batch_size,
        }
        if self.dino_token is not None:
            dd["dino_token"] = self.dino_token
        if self.nfaces is not None:
            dd["nfaces"] = self.nfaces
        return dd


@dataclass(frozen=True)
class LatVaeDecodeParams(RuntimeParams):
    """Placeholder for future LatVae decode-time runtime parameters."""

    @classmethod
    def from_geo_config(cls, geo_config: Dict[str, Any]) -> "LatVaeDecodeParams":
        return cls()

    @classmethod
    def from_dict(cls, data: Optional[Dict[str, Any]]) -> "LatVaeDecodeParams":
        return cls()


@dataclass(frozen=True)
class MeshOutput:
    vertices: "torch.Tensor"
    triangles: "torch.Tensor"


@dataclass(frozen=True)
class Image23DRuntimeParams:
    """Pipeline-level runtime defaults for all stages."""

    pcd_dit: FlowMatchSamplingParams = field(default_factory=FlowMatchSamplingParams)
    pcd_vae: PcdVaeDecodeParams = field(default_factory=PcdVaeDecodeParams)
    pcd_rep: PcdRepGeometryParams = field(default_factory=PcdRepGeometryParams)
    lat_dit: FlowMatchSamplingParams = field(default_factory=FlowMatchSamplingParams)
    lat_vae: LatVaeDecodeParams = field(default_factory=LatVaeDecodeParams)

    @classmethod
    def from_geo_config(cls, geo_config: Dict[str, Any]) -> "Image23DRuntimeParams":
        gen = geo_config.get("generators", {})
        return cls(
            pcd_dit=FlowMatchSamplingParams.from_yaml_section(gen.get("pcd_dit", {})),
            pcd_vae=PcdVaeDecodeParams.from_geo_config(geo_config),
            pcd_rep=PcdRepGeometryParams.from_geo_config(geo_config),
            lat_dit=FlowMatchSamplingParams.from_yaml_section(gen.get("lat_dit", {})),
            lat_vae=LatVaeDecodeParams.from_geo_config(geo_config),
        )

    @classmethod
    def from_dict(cls, data: Optional[Dict[str, Any]]) -> "Image23DRuntimeParams":
        if not data:
            return cls.from_geo_config({})
        return cls(
            pcd_dit=FlowMatchSamplingParams.from_dict(data.get("pcd_dit")),
            pcd_vae=PcdVaeDecodeParams.from_dict(data.get("pcd_vae")),
            pcd_rep=PcdRepGeometryParams.from_dict(data.get("pcd_rep")),
            lat_dit=FlowMatchSamplingParams.from_dict(data.get("lat_dit")),
            lat_vae=LatVaeDecodeParams.from_dict(data.get("lat_vae")),
        )

    def to_dict(self) -> Dict[str, Any]:
        return {
            "pcd_dit": asdict(self.pcd_dit),
            "pcd_vae": asdict(self.pcd_vae),
            "pcd_rep": asdict(self.pcd_rep),
            "lat_dit": asdict(self.lat_dit),
            "lat_vae": asdict(self.lat_vae),
        }

    def merge(
        self,
        other: Optional["Image23DRuntimeParams"] = None,
        **kwargs: Any,
    ) -> "Image23DRuntimeParams":
        base = self
        if other is not None:
            base = Image23DRuntimeParams(
                pcd_dit=base.pcd_dit.merge(other.pcd_dit),
                pcd_vae=base.pcd_vae.merge(other.pcd_vae),
                pcd_rep=base.pcd_rep.merge(other.pcd_rep),
                lat_dit=base.lat_dit.merge(other.lat_dit),
                lat_vae=base.lat_vae.merge(other.lat_vae),
            )
        updates: Dict[str, Any] = {}
        for stage in ("pcd_dit", "pcd_vae", "pcd_rep", "lat_dit", "lat_vae"):
            prefixed = {
                k[len(stage) + 1 :]: v
                for k, v in kwargs.items()
                if k.startswith(f"{stage}_") and v is not None
            }
            if prefixed:
                updates[stage] = getattr(base, stage).merge(None, **prefixed)
        if updates:
            base = replace(base, **updates)
        return base


def resolve(
    default: T,
    override: Optional[T] = None,
    **kwargs: Any,
) -> T:
    """Merge runtime params: default <- override <- kwargs (None values ignored)."""
    return default.merge(override, **kwargs)


def wire_module_runtimes(modules: Dict[str, Any], runtime: Image23DRuntimeParams) -> None:
    """Attach pipeline runtime defaults to each model submodule."""
    modules["pcd_dit"]._default_runtime = runtime.pcd_dit
    modules["pcd_vae"]._default_runtime = runtime.pcd_vae
    modules["pcd_rep"]._default_runtime = runtime.pcd_rep
    modules["lat_dit"]._default_runtime = runtime.lat_dit
    modules["lat_vae"]._default_runtime = runtime.lat_vae


In [ ]:
%%writefile /kaggle/working/stub/kandi3d_image23d/lat_dit_protocol.py
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import asdict, is_dataclass
import time
from typing import Any, Dict, Optional, Union

import torch

from kandi3d_image23d.runtime import FlowMatchSamplingParams, LatDitConditioning


_OMIT = object()


def _json_safe(value: Any) -> Any:
    """Convert serializable values and omit tensors/opaque runtime objects."""
    if value is None or isinstance(value, (bool, int, float, str)):
        return value
    if is_dataclass(value):
        return _json_safe(asdict(value))
    if isinstance(value, dict):
        result = {}
        for key, item in value.items():
            converted = _json_safe(item)
            if converted is not _OMIT:
                result[str(key)] = converted
        return result
    if isinstance(value, (list, tuple)):
        result = []
        for item in value:
            converted = _json_safe(item)
            if converted is not _OMIT:
                result.append(converted)
        return result
    return _OMIT


class LatDitGenerator(ABC):
    """
    Minimal contract for the hackathon "lat_dit track".

    Participants implement this interface so their model can be swapped in without touching the rest
    of the pipeline (cached voxels + cached DINOv3 conditioning + fixed lat_vae decoder).

    `generate()` is a mandatory, non-overridable wrapper. Implement `generate_impl()` only.
    The wrapper synchronizes CUDA, measures generation wall time and stores JSON-safe metadata in
    `last_run_metadata`. Cached voxel indices and DINO tensors are never recorded.
    """

    last_run_metadata: Dict[str, Any]

    def __init_subclass__(cls, **kwargs: Any) -> None:
        super().__init_subclass__(**kwargs)
        if "generate" in cls.__dict__:
            raise TypeError(
                f"{cls.__name__} must not override LatDitGenerator.generate(); "
                "implement generate_impl() instead"
            )

    @classmethod
    @abstractmethod
    def from_pretrained(
        cls,
        pretrained_model_name_or_path: str,
        device: Optional[Union[torch.device, str, int]] = None,
        torch_dtype: Optional[torch.dtype] = None,
        **kwargs,
    ) -> "LatDitGenerator":
        raise NotImplementedError

    def generate(
        self,
        vox_idxs: list,
        *,
        conditioning: LatDitConditioning,
        sampling: Optional[FlowMatchSamplingParams] = None,
        generator: Optional[torch.Generator] = None,
        **sampling_kwargs,
    ) -> list:
        """
        Args:
          vox_idxs: list of tensors with shape [N, 3] (integer voxel indices).
          conditioning: DINOv3 conditioning bundle.
          sampling: Flow-matching sampling params.
          generator: optional RNG.

        Returns:
          list of sparse latent feature tensors compatible with `LatVaeModel`:
          each element corresponds to one batch entry and has shape [N, latent_dim].
        """

        resolved_sampling = sampling or getattr(
            self, "_default_runtime", FlowMatchSamplingParams()
        )
        serializable_kwargs = _json_safe(sampling_kwargs)
        metadata: Dict[str, Any] = {
            "generator_class": f"{type(self).__module__}.{type(self).__qualname__}",
            "inputs": {
                "sampling": _json_safe(resolved_sampling),
                "sampling_kwargs": serializable_kwargs,
                "batch_size": int(conditioning.batch_size),
                "num_voxels": [int(vox.shape[0]) for vox in vox_idxs],
                "generator_seed": int(generator.initial_seed()) if generator is not None else None,
            },
        }

        cuda_device = None
        for vox in vox_idxs:
            if torch.is_tensor(vox) and vox.is_cuda:
                cuda_device = vox.device
                break

        if cuda_device is not None:
            torch.cuda.synchronize(cuda_device)
        started = time.perf_counter()
        try:
            result = self.generate_impl(
                vox_idxs,
                conditioning=conditioning,
                sampling=sampling,
                generator=generator,
                **sampling_kwargs,
            )
            if cuda_device is not None:
                torch.cuda.synchronize(cuda_device)
            metadata["elapsed_seconds"] = float(time.perf_counter() - started)
            metadata["status"] = "ok"
            self.last_run_metadata = metadata
            return result
        except Exception as exc:
            if cuda_device is not None:
                torch.cuda.synchronize(cuda_device)
            metadata["elapsed_seconds"] = float(time.perf_counter() - started)
            metadata["status"] = "error"
            metadata["error_type"] = type(exc).__name__
            self.last_run_metadata = metadata
            raise

    @abstractmethod
    def generate_impl(
        self,
        vox_idxs: list,
        *,
        conditioning: LatDitConditioning,
        sampling: Optional[FlowMatchSamplingParams] = None,
        generator: Optional[torch.Generator] = None,
        **sampling_kwargs: Any,
    ) -> list:
        """Generate sparse latents; timing and metadata are handled by `generate()`."""
        raise NotImplementedError


__all__ = ["LatDitGenerator"]



In [ ]:
%%writefile /kaggle/working/stub/kandi3d_image23d/__init__.py
# stub: only the official generator protocol (the real package needs the whole 3D stack)
from kandi3d_image23d.lat_dit_protocol import LatDitGenerator  # noqa: F401


In [ ]:
%%writefile /kaggle/working/k3d/calib/k3d_data.py
"""Download helpers for the competition data hosted on sbercloud OBS (S3-compatible, supports HTTP Range).

* extract_from_remote_zip(url, dest, member_filter): pulls only the selected members of a remote zip
  via Range requests (falls back to a full download when the server ignores Range).
* stream_tar_members(url, dest, member_filter, stop_after): streams a remote tar and keeps matching
  members; can stop early, so a 16 GB chunk does not have to be stored.
"""

from __future__ import annotations

import io
import os
import shutil
import struct
import tarfile
import time
import zipfile
from pathlib import Path
from typing import Callable, Iterable, List, Optional

import requests

BASE = "https://rndml-team-xr.obs.ru-moscow-1.hc.sbercloud.ru/mazurov/tmp_data/AIJC_data"
URLS = {
    "code": f"{BASE}/3dgen-main.zip",
    "weights": f"{BASE}/kandi3d_weights_aijc_kit.zip",
    "public_test": f"{BASE}/objhq_splits_clean/public_test.zip",
    "train": f"{BASE}/objhq_splits_clean/train_archives/train_part_{{:03d}}.tar",
}


def _session() -> requests.Session:
    s = requests.Session()
    s.headers["User-Agent"] = "k3d-kaggle/1.0"
    return s


class HttpRangeReader(io.RawIOBase):
    """Seekable read-only file over HTTP Range requests with a simple read-ahead buffer."""

    def __init__(self, url: str, chunk: int = 16 << 20, session: Optional[requests.Session] = None, retries: int = 5):
        self.url, self.chunk, self.retries = url, chunk, retries
        self.s = session or _session()
        r = self.s.head(url, allow_redirects=True, timeout=60)
        r.raise_for_status()
        self.size = int(r.headers["Content-Length"])
        self.pos = 0
        self._buf, self._buf_start = b"", -1

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else self.pos + off if whence == 1 else self.size + off
        return self.pos

    def fetch(self, start: int, end_incl: int) -> bytes:
        for attempt in range(self.retries):
            try:
                r = self.s.get(self.url, headers={"Range": f"bytes={start}-{end_incl}"}, timeout=300)
                if r.status_code != 206:
                    raise IOError(f"server ignored Range (status {r.status_code})")
                data = r.content
                if len(data) != end_incl - start + 1:
                    raise IOError("short read")
                return data
            except (requests.RequestException, IOError):
                if attempt == self.retries - 1:
                    raise
                time.sleep(2 * (attempt + 1))
        raise IOError("unreachable")

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = max(0, min(n, self.size - self.pos))
        if n == 0:
            return b""
        bs = self._buf_start
        if bs >= 0 and bs <= self.pos and self.pos + n <= bs + len(self._buf):
            o = self.pos - bs
            data = self._buf[o:o + n]
        elif n >= self.chunk:
            data = self.fetch(self.pos, self.pos + n - 1)
        else:
            end = min(self.size, self.pos + self.chunk) - 1
            self._buf, self._buf_start = self.fetch(self.pos, end), self.pos
            data = self._buf[:n]
        self.pos += len(data)
        return data

    def readinto(self, b):
        data = self.read(len(b))
        b[:len(data)] = data
        return len(data)


def _stream_range_to_file(sess, url, start, length, dst: Path, chunk=8 << 20):
    end = start + length - 1
    with sess.get(url, headers={"Range": f"bytes={start}-{end}"}, stream=True, timeout=600) as r:
        if r.status_code != 206:
            raise IOError(f"server ignored Range (status {r.status_code})")
        with open(dst, "wb") as f:
            for part in r.iter_content(chunk_size=chunk):
                f.write(part)
    if dst.stat().st_size != length:
        raise IOError(f"size mismatch for {dst}")


def _stream_deflate_to_file(sess, url, start, csize, usize, dst: Path, chunk=8 << 20):
    import zlib
    d = zlib.decompressobj(-15)
    end = start + csize - 1
    with sess.get(url, headers={"Range": f"bytes={start}-{end}"}, stream=True, timeout=600) as r:
        if r.status_code != 206:
            raise IOError(f"server ignored Range (status {r.status_code})")
        with open(dst, "wb") as f:
            for part in r.iter_content(chunk_size=chunk):
                f.write(d.decompress(part))
            f.write(d.flush())
    if dst.stat().st_size != usize:
        raise IOError(f"size mismatch for {dst}")


def download(url: str, dst: Path) -> Path:
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_suffix(dst.suffix + ".part")
    with _session().get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            shutil.copyfileobj(r.raw, f, length=16 << 20)
    tmp.rename(dst)
    return dst


def list_remote_zip(url: str) -> List[zipfile.ZipInfo]:
    with zipfile.ZipFile(HttpRangeReader(url, chunk=1 << 20)) as zf:
        return zf.infolist()


def _target_name(name: str, strip_prefix: Optional[str], rename: Optional[Callable[[str], str]]) -> str:
    if rename is not None:
        return rename(name)
    if strip_prefix and name.startswith(strip_prefix):
        return name[len(strip_prefix):]
    return name


def extract_from_remote_zip(url: str, dest: Path, member_filter: Callable[[str], bool],
                            strip_prefix: Optional[str] = None, verbose: bool = True,
                            rename: Optional[Callable[[str], str]] = None) -> List[Path]:
    """Extract matching members; STORED members are streamed in one Range request each."""
    dest = Path(dest)
    out: List[Path] = []
    sess = _session()
    reader = HttpRangeReader(url, chunk=16 << 20, session=sess)
    with zipfile.ZipFile(reader) as zf:
        members = [m for m in zf.infolist() if not m.is_dir() and member_filter(m.filename)]
        total = sum(m.file_size for m in members)
        if verbose:
            print(f"[zip] {url.rsplit('/', 1)[-1]}: {len(members)} members, {total / 2**30:.2f} GiB")
        for k, m in enumerate(members):
            name = _target_name(m.filename, strip_prefix, rename)
            dst = dest / name
            if dst.exists() and dst.stat().st_size == m.file_size:
                out.append(dst)
                continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            big = m.file_size > (64 << 20)
            if big and m.compress_type in (zipfile.ZIP_STORED, zipfile.ZIP_DEFLATED):
                hdr = reader.fetch(m.header_offset, m.header_offset + 29)
                n_name, n_extra = struct.unpack("<HH", hdr[26:30])
                data_start = m.header_offset + 30 + n_name + n_extra
                if m.compress_type == zipfile.ZIP_STORED:
                    _stream_range_to_file(sess, url, data_start, m.file_size, dst)
                else:
                    _stream_deflate_to_file(sess, url, data_start, m.compress_size, m.file_size, dst)
            else:
                with zf.open(m) as src, open(dst, "wb") as f:
                    shutil.copyfileobj(src, f, length=8 << 20)
            out.append(dst)
            if verbose and (m.file_size > (256 << 20) or (k + 1) % 50 == 0 or k + 1 == len(members)):
                print(f"[zip]   {k + 1}/{len(members)} {name} ({m.file_size / 2**20:.0f} MiB)")
    return out


def extract_zip_any(url: str, dest: Path, member_filter: Callable[[str], bool], strip_prefix: Optional[str] = None,
                    tmp_dir: Path = Path("/tmp/k3d_dl"), rename: Optional[Callable[[str], str]] = None) -> List[Path]:
    """Range-based extraction, or full download + local extraction if the server does not support Range."""
    try:
        return extract_from_remote_zip(url, dest, member_filter, strip_prefix, rename=rename)
    except IOError as exc:
        print(f"[zip] range extraction failed ({exc}); downloading the whole archive")
        local = download(url, Path(tmp_dir) / url.rsplit("/", 1)[-1])
        out = []
        with zipfile.ZipFile(local) as zf:
            for m in zf.infolist():
                if m.is_dir() or not member_filter(m.filename):
                    continue
                dst = Path(dest) / _target_name(m.filename, strip_prefix, rename)
                dst.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(m) as src, open(dst, "wb") as f:
                    shutil.copyfileobj(src, f, length=8 << 20)
                out.append(dst)
        local.unlink(missing_ok=True)
        return out


def stream_tar_members(url: str, dest: Path, member_filter: Callable[[str], bool],
                       stop_after_dirs: Optional[int] = None, verbose: bool = True) -> List[Path]:
    """Stream a remote tar, keep matching members. stop_after_dirs: stop once that many sample ids were seen."""
    dest = Path(dest)
    out, seen = [], []
    with _session().get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        r.raw.decode_content = True
        with tarfile.open(fileobj=r.raw, mode="r|*") as tf:
            for m in tf:
                parts = Path(m.name).parts
                sid = parts[1] if len(parts) > 2 else (parts[0] if parts else "")
                if sid and (not seen or seen[-1] != sid):
                    if stop_after_dirs is not None and len(seen) >= stop_after_dirs:
                        break
                    seen.append(sid)
                if not m.isfile() or not member_filter(m.name):
                    continue
                dst = dest / m.name
                dst.parent.mkdir(parents=True, exist_ok=True)
                src = tf.extractfile(m)
                with open(dst, "wb") as f:
                    shutil.copyfileobj(src, f, length=8 << 20)
                out.append(dst)
                if verbose and len(out) % 200 == 0:
                    print(f"[tar] {len(out)} files, {len(seen)} samples")
    if verbose:
        print(f"[tar] done: {len(out)} files from {len(seen)} samples")
    return out


In [ ]:
%%writefile /kaggle/working/sub7/solution.py
"""K3D student: a tiny per-voxel network that replaces the lat_dit diffusion model (AIJ 2026, 3D DIT Distillation).

Every voxel's 16-channel latent is predicted by a small MLP from features of the voxel set only:
  occ  124  occupancy of the 5x5x5 neighbourhood (centre excluded), offsets ordered dx, dy, dz = -2..2
  pos    6  voxel / 64 and (voxel - 31.5) / (31.5 + 1e-6), rounded to fp16 (the training features used the bbox centre
            and half of the largest extent; every training and public-test object is centred with half-extent 31.5,
            so the fixed values are identical there)
  (ctx 125  occupied-voxel counts / 64 of the 5x5x5 neighbourhood of 4x4x4 blocks: reference path only)
MLP 130 -> H -> H -> 16 with SiLU, trained to regress the teacher's one-step conditional prediction at t=1 (x0c).
The output deviations are rescaled per channel around the training mean (variance matching folded into the last
layer). All input-dependent computation runs inside generate_impl; from_pretrained only loads and repacks weights.

Two implementations of the same function:
  _Reference  plain PyTorch (any device); fallback and ground truth for the checks;
  _Engine     CUDA kernels launched through the CUDA driver API (no compiler needed at run time), engine v7:
                A  occupancy bit rows (one 64-bit row per (x, y)), atomics; clears the next call's buffers
                C  per-voxel features + MLP on tensor cores (fp16 in, fp32 accumulate) over 16-voxel tiles spread
                   evenly over at most one CTA per SM; the packed weights are loaded once per CTA with all loads in
                   flight and read from shared memory, activations stay in registers; the occupancy bits are packed so
                   that layer 1 needs no per-bit unpacking (W1 columns permuted at load)
              launched as two kernels ("split"), or one kernel with a grid barrier (launched cooperatively, "coop",
              or as a plain launch, "fused"). Kernels come from cubins in weights/ (built from the source below) or are
              compiled with NVRTC at load. The engine is used only after it matches the reference on a self-check at
              load time; the fastest launch mode is chosen by a benchmark with flushed caches.
generate_impl itself is one C call (k3d_fast7 extension, CPython limited API): tensor checks, output allocation and
the launches, without per-call Python; the Python path stays as its fallback and as the path of the checks.
Environment overrides: K3D_ENGINE = auto | split | coop | fused | off, K3D_SHIM = auto | off, K3D_FAST = auto | off,
K3D_VM = global | none | object, K3D_GRID_CAP = CTAs of the mlp / fused kernels (default: SM count), K3D_VERBOSE = 0 | 1.
"""

from __future__ import annotations

import ctypes
import ctypes.util
import glob
import hashlib
import os
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import torch

from kandi3d_image23d import LatDitGenerator

GR = 68                # occupancy grid side with a 2-voxel margin
NROW = GR * GR         # 64-bit occupancy rows (bit z) per (x, y)
NBLK = 4096            # 16^3 coarse 4x4x4 blocks
_VERBOSE = os.environ.get("K3D_VERBOSE", "1") != "0"


def _log(msg: str) -> None:
    if _VERBOSE:
        print(f"[k3d] {msg}", flush=True)


# ----------------------------------------------------------------------------------------------------------------
# CUDA source (no headers needed; compiles with NVRTC or nvcc for sm_75+)
# ----------------------------------------------------------------------------------------------------------------
CUDA_SRC = r"""
// K3D engine v7. Every kernel runs once per call with cold caches (and, in the evaluation loop, most likely on an idle,
// down-clocked GPU), so the design minimises serial memory round trips and the instructions per voxel:
//  - the packed weights (one blob) are loaded with all loads in flight at kernel start and parked in shared memory;
//  - 16-voxel tiles are spread evenly over at most one CTA (16 warps) per SM: tile = cta + grid * (warp + 16 * iter);
//  - the 25 occupancy rows of a voxel are split between the two half-warps (13 + 12 rows, one round trip);
//  - the occupancy bits are packed so that every A-fragment word of layer 1 is (word & mask) * const: the input columns
//    of W1 are permuted at packing time to match (no per-bit unpacking);
//  - B fragments of two n-tiles and the (pre-duplicated) biases come with one 128-bit shared load each;
//  - hidden pre-activations arrive halved (x / 2 folded into W1, b1, W2, b2): SiLU(x) = h + h * tanh(h);
//  - layer activations stay in registers (C fragments of one layer are the A fragments of the next);
//  - the position features use the fixed grid centre (every known object is centred with half-extent 31.5).
typedef unsigned int u32;
typedef unsigned long long u64;

#define GR 68
#define NROW 4624
#define NWARP 16
#define NTC 512         // threads per CTA of the mlp / fused kernels

struct P {
    const int* vox; float* out; u64* rows; u32* bar; const uint4* wts;
    int n; int par; int nwl; int vs0; int vs1; int pad;   // vox element (i, c) at vox[i * vs0 + c * vs1]
};

// weight blob layout (u32 words), q = pair of n-tiles (2q, 2q + 1):
// W1 [8 kt][NQ][32 lanes][4] | W1P [NQ][32][2] | B1 [NT][4 t][4] | W2 [KT][NQ][32][4] | B2 [NT][4][4] | W3 [KT][32][4] |
// B3 [2][4][4]; a B pair holds (b0, b1) of n-tile 2q then of 2q + 1; a bias quad is (b[n], b[n + 1], b[n], b[n + 1]),
// n = 8 j + 2 t (the C fragment of the lane)
template <int H> struct WL {
    static constexpr int NT = H / 8, KT = H / 16, NQ = NT / 2;
    static constexpr int W1 = 0, W1P = W1 + 8 * NQ * 128, B1 = W1P + NQ * 64, W2 = B1 + NT * 16, B2 = W2 + KT * NQ * 128,
                         W3 = B2 + NT * 16, B3 = W3 + KT * 128, TOT = B3 + 32, NV = TOT / 4, WPT = (NV + NTC - 1) / NTC;
};

__device__ __forceinline__ int clamp63(int v) { return min(max(v, 0), 63); }

__device__ __forceinline__ void prefetch_l2(const void* a) { asm volatile("prefetch.global.L2 [%0];" :: "l"(a)); }

__device__ __forceinline__ u32 f2h2(float lo, float hi) {  // round-to-nearest fp16 pair
    unsigned short a, b;
    asm("cvt.rn.f16.f32 %0, %1;" : "=h"(a) : "f"(lo));
    asm("cvt.rn.f16.f32 %0, %1;" : "=h"(b) : "f"(hi));
    return (u32)a | ((u32)b << 16);
}

__device__ __forceinline__ float silu_h(float h) {         // SiLU(2 h) = h + h * tanh(h)
    float t;
    asm("tanh.approx.f32 %0, %1;" : "=f"(t) : "f"(h));
    return fmaf(h, t, h);
}

__device__ __forceinline__ void mma_k16(float* d, const u32* a, u32 b0, u32 b1) {
#if __CUDA_ARCH__ >= 800
    asm("mma.sync.aligned.m16n8k16.row.col.f32.f16.f16.f32 {%0,%1,%2,%3}, {%4,%5,%6,%7}, {%8,%9}, {%0,%1,%2,%3};"
        : "+f"(d[0]), "+f"(d[1]), "+f"(d[2]), "+f"(d[3])
        : "r"(a[0]), "r"(a[1]), "r"(a[2]), "r"(a[3]), "r"(b0), "r"(b1));
#else
    asm("mma.sync.aligned.m16n8k8.row.col.f32.f16.f16.f32 {%0,%1,%2,%3}, {%4,%5}, {%6}, {%0,%1,%2,%3};"
        : "+f"(d[0]), "+f"(d[1]), "+f"(d[2]), "+f"(d[3]) : "r"(a[0]), "r"(a[1]), "r"(b0));
    asm("mma.sync.aligned.m16n8k8.row.col.f32.f16.f16.f32 {%0,%1,%2,%3}, {%4,%5}, {%6}, {%0,%1,%2,%3};"
        : "+f"(d[0]), "+f"(d[1]), "+f"(d[2]), "+f"(d[3]) : "r"(a[2]), "r"(a[3]), "r"(b1));
#endif
}

__device__ __forceinline__ void mma_k8(float* d, u32 a0, u32 a1, u32 b0) {
    asm("mma.sync.aligned.m16n8k8.row.col.f32.f16.f16.f32 {%0,%1,%2,%3}, {%4,%5}, {%6}, {%0,%1,%2,%3};"
        : "+f"(d[0]), "+f"(d[1]), "+f"(d[2]), "+f"(d[3]) : "r"(a0), "r"(a1), "r"(b0));
}

// A-fragment word of layer 1: bits j and j + 16 of an occupancy word -> half2 (1.0 or 0.0 each); w5 = w >> 5
__device__ __forceinline__ u32 expand(u32 w, u32 w5, int j) {
    return j <= 10 ? (w & (0x10001u << j)) * (15u << (10 - j)) : (w5 & (0x10001u << (j - 5))) * (15u << (15 - j));
}

// SiLU of two adjacent n-tiles (C fragments of halved pre-activations) -> one A fragment (k16) of the next layer
__device__ __forceinline__ void act(u32* a, const float* c0, const float* c1) {
    a[0] = f2h2(silu_h(c0[0]), silu_h(c0[1]));
    a[1] = f2h2(silu_h(c0[2]), silu_h(c0[3]));
    a[2] = f2h2(silu_h(c1[0]), silu_h(c1[1]));
    a[3] = f2h2(silu_h(c1[2]), silu_h(c1[3]));
}

__device__ __forceinline__ void cinit(float* c, const float4 q) { c[0] = q.x; c[1] = q.y; c[2] = q.z; c[3] = q.w; }

// ---- phase A: occupancy rows (bit z of row (x + 2, y + 2)) of this call's parity; clears the other parity
__device__ __forceinline__ void stamp(const P& p, int tid, int nthr) {
    u64* rows = p.rows + (size_t)p.par * NROW;
    uint4* rn = (uint4*)(p.rows + (size_t)(p.par ^ 1) * NROW);
#pragma unroll 1
    for (int j = tid; j < NROW / 16; j += nthr) prefetch_l2(rows + 16 * j);   // rows the gather will read
#pragma unroll 1
    for (int j = tid; j < NROW / 2; j += nthr) rn[j] = make_uint4(0u, 0u, 0u, 0u);
    if (tid == 0) p.bar[(p.par ^ 1) * 32] = 0u;
#pragma unroll 1
    for (int i = tid; i < p.n; i += nthr) {
        const int* q = p.vox + (size_t)i * p.vs0;
        const int x = clamp63(__ldg(q)), y = clamp63(__ldg(q + p.vs1)), z = clamp63(__ldg(q + 2 * p.vs1));
        atomicOr(rows + (x + 2) * GR + (y + 2), 1ull << z);
    }
}

// grid barrier (fused kernel): bar[0] counts arrivals (cleared for the next call by stamp); bar[1] is set if a barrier
// ever times out (never expected; the self-check then rejects the mode)
__device__ __forceinline__ void grid_sync(u32* bar, u32 target) {
    __syncthreads();
    if (threadIdx.x == 0) {
        __threadfence();
        atomicAdd(bar, 1u);
        volatile u32* vb = bar;
        unsigned spins = 0;
        while ((int)(*vb - target) < 0) {
            __nanosleep(32);
            if (++spins > (1u << 22)) { atomicExch(bar + 1, 1u); break; }
        }
        __threadfence();
    }
    __syncthreads();
}

template <int H> __device__ __forceinline__ void wload(const P& p, uint4* v) {
#pragma unroll
    for (int k = 0; k < WL<H>::WPT; k++) {
        const int i = threadIdx.x + k * NTC;
        if (i < WL<H>::NV) v[k] = __ldg(p.wts + i);
    }
}

template <int H> __device__ __forceinline__ void wstore(u32* sW, const uint4* v) {
#pragma unroll
    for (int k = 0; k < WL<H>::WPT; k++) {
        const int i = threadIdx.x + k * NTC;
        if (i < WL<H>::NV) ((uint4*)sW)[i] = v[k];
    }
}

// bits z - 2 .. z + 2 of an occupancy row (zero outside 0..63)
__device__ __forceinline__ u32 zbits(u64 w, int z) {
    return (u32)((z >= 2) ? (w >> (z - 2)) : (w << (2 - z))) & 31u;
}

// ---- features of the voxel of lane v of a tile, built by both half-warps h (one round trip: all rows in flight):
// half 0 packs rows (dx, dy) = (-2..-1, -2..2), (0, -2), (0, -1) and the centre row (0, 0) without the centre bit into
// occupancy word pair 0..1; half 1 packs rows (1..2, -2..2), (0, 1), (0, 2) into words 2..3 (5 bits dz = -2..2 per row,
// in that order). Position values (x / 64, (x - 31.5) / (31.5 + 1e-6), as fp16) go to sP.
#define PRC __int_as_float(0x3d020820)
__device__ __forceinline__ void gather(const u64* rows, u32* sO, u32* sP, int v, int h, bool on, int x, int y, int z) {
    u64 acc = 0ull;
    u32 q0 = 0u, q1 = 0u, q2 = 0u;
    if (on) {
        const u64* r0 = rows + x * GR + y;     // row (x + dx + 2, y + dy + 2) = r0[(dx + 2) * GR + dy + 2]
        const u64* rA = r0 + h * (3 * GR);
        const u64* rB = r0 + 2 * GR + h * 3;
        u64 w[13];
#pragma unroll
        for (int i = 0; i < 10; i++) w[i] = __ldcg(rA + (i / 5) * GR + (i % 5));
        w[10] = __ldcg(rB);
        w[11] = __ldcg(rB + 1);
        w[12] = __ldcg(r0 + 2 * GR + 2);
#pragma unroll
        for (int i = 0; i < 12; i++) acc |= (u64)zbits(w[i], z) << (5 * i);
        u32 f = zbits(w[12], z);
        f = ((f & 3u) | ((f >> 1) & 12u)) & (h ? 0u : 15u);  // centre voxel excluded; half 0 only
        acc |= (u64)f << 60;
        const float fx = (float)x, fy = (float)y, fz = (float)z;
        q0 = f2h2(fx * 0.015625f, fy * 0.015625f);
        q1 = f2h2(fz * 0.015625f, (fx - 31.5f) * PRC);
        q2 = f2h2((fy - 31.5f) * PRC, (fz - 31.5f) * PRC);
    }
    *(uint2*)(sO + v * 4 + 2 * h) = make_uint2((u32)acc, (u32)(acc >> 32));
    *(uint2*)(sP + v * 4 + 2 * h) = h ? make_uint2(q2, 0u) : make_uint2(q0, q1);
}

// voxel coordinates of lane v of a tile (every lane: both half-warps work on the same 16 voxels)
__device__ __forceinline__ bool vload(const P& p, int tile, int v, int& x, int& y, int& z) {
    const int i = tile * 16 + v;
    const bool on = i < p.n;
    if (on) {
        const int* q = p.vox + (size_t)i * p.vs0;
        x = clamp63(__ldg(q)); y = clamp63(__ldg(q + p.vs1)); z = clamp63(__ldg(q + 2 * p.vs1));
    }
    return on;
}

// ---- per-warp tiles of 16 voxels: features -> MLP 130 -> H -> H -> 16 (SiLU) on tensor cores, weights in shared memory.
// Every warp first issues the voxel loads of its first tile, then completes the CTA prologue:
//   MODE 0 (mlp kernel of the split launch): park the weights in shared memory + __syncthreads;
//   MODE 1 (fused kernel): grid barrier (the stamp phase and the weight parking happened before).
template <int H, int MODE>
__device__ __forceinline__ void tiles(const P& p, u32* sW, const uint4* wv, u32* sO, u32* sP) {
    typedef WL<H> L;
    constexpr int NT = L::NT, KT = L::KT, NQ = L::NQ;
    const int lane = threadIdx.x & 31, g = lane >> 2, t = lane & 3, v = lane & 15, h = lane >> 4;
    const u64* rows = p.rows + (size_t)p.par * NROW;
    const uint4* w1 = (const uint4*)(sW + L::W1);
    const uint2* w1p = (const uint2*)(sW + L::W1P);
    const float4* b1 = (const float4*)(sW + L::B1);
    const uint4* w2 = (const uint4*)(sW + L::W2);
    const float4* b2 = (const float4*)(sW + L::B2);
    const uint4* w3 = (const uint4*)(sW + L::W3);
    const float4* b3 = (const float4*)(sW + L::B3);
    const int step = gridDim.x * NWARP;
    int tile = blockIdx.x + gridDim.x * (threadIdx.x >> 5);
    int x = 0, y = 0, z = 0;
    bool on = vload(p, tile, v, x, y, z);
    if (MODE == 0) { wstore<H>(sW, wv); __syncthreads(); }   // prologue outside the loop: wv dies here
    else grid_sync(p.bar + p.par * 32, gridDim.x);
#pragma unroll 1
    while (tile * 16 < p.n) {
        gather(rows, sO, sP, v, h, on, x, y, z);
        __syncwarp();
        // layer 1: 8 k-tiles of occupancy bits (A words straight from the packed words) + one k8 tile of positions
        const u32 wa = sO[g * 4 + t], wb = sO[(g + 8) * 4 + t], wa5 = wa >> 5, wb5 = wb >> 5;
        float acc[NT][4];
#pragma unroll
        for (int j = 0; j < NT; j++) cinit(acc[j], b1[j * 4 + t]);
#pragma unroll
        for (int kt = 0; kt < 8; kt++) {
            u32 a[4];
            a[0] = expand(wa, wa5, 2 * kt); a[1] = expand(wb, wb5, 2 * kt);
            a[2] = expand(wa, wa5, 2 * kt + 1); a[3] = expand(wb, wb5, 2 * kt + 1);
#pragma unroll
            for (int q = 0; q < NQ; q++) {
                const uint4 b = w1[(kt * NQ + q) * 32 + lane];
                mma_k16(acc[2 * q], a, b.x, b.y);
                mma_k16(acc[2 * q + 1], a, b.z, b.w);
            }
        }
        {
            const u32 pa = t < 3 ? sP[g * 4 + t] : 0u, pb = t < 3 ? sP[(g + 8) * 4 + t] : 0u;
#pragma unroll
            for (int q = 0; q < NQ; q++) {
                const uint2 b = w1p[q * 32 + lane];
                mma_k8(acc[2 * q], pa, pb, b.x);
                mma_k8(acc[2 * q + 1], pa, pb, b.y);
            }
        }
        u32 a2[KT][4];
#pragma unroll
        for (int k = 0; k < KT; k++) act(a2[k], acc[2 * k], acc[2 * k + 1]);
        // layer 2
        float acc2[NT][4];
#pragma unroll
        for (int j = 0; j < NT; j++) cinit(acc2[j], b2[j * 4 + t]);
#pragma unroll
        for (int k = 0; k < KT; k++) {
#pragma unroll
            for (int q = 0; q < NQ; q++) {
                const uint4 b = w2[(k * NQ + q) * 32 + lane];
                mma_k16(acc2[2 * q], a2[k], b.x, b.y);
                mma_k16(acc2[2 * q + 1], a2[k], b.z, b.w);
            }
        }
        u32 a3[KT][4];
#pragma unroll
        for (int k = 0; k < KT; k++) act(a3[k], acc2[2 * k], acc2[2 * k + 1]);
        // layer 3 (variance matching folded in)
        float o[2][4];
        cinit(o[0], b3[t]);
        cinit(o[1], b3[4 + t]);
#pragma unroll
        for (int k = 0; k < KT; k++) {
            const uint4 b = w3[k * 32 + lane];
            mma_k16(o[0], a3[k], b.x, b.y);
            mma_k16(o[1], a3[k], b.z, b.w);
        }
        const int ra = tile * 16 + g, rb = ra + 8;
        if (ra < p.n) {
            *(float2*)(p.out + (size_t)ra * 16 + 2 * t) = make_float2(o[0][0], o[0][1]);
            *(float2*)(p.out + (size_t)ra * 16 + 8 + 2 * t) = make_float2(o[1][0], o[1][1]);
        }
        if (rb < p.n) {
            *(float2*)(p.out + (size_t)rb * 16 + 2 * t) = make_float2(o[0][2], o[0][3]);
            *(float2*)(p.out + (size_t)rb * 16 + 8 + 2 * t) = make_float2(o[1][2], o[1][3]);
        }
        __syncwarp();
        tile += step;
        on = vload(p, tile, v, x, y, z);
    }
}

#define TID (blockIdx.x * blockDim.x + threadIdx.x)
#define NTH (gridDim.x * blockDim.x)

template <int H> __device__ __forceinline__ void mlp_kernel(const P& p) {
    __shared__ __align__(16) u32 sW[WL<H>::TOT];
    __shared__ __align__(16) u32 sO[NWARP][64];
    __shared__ __align__(16) u32 sP[NWARP][64];
    uint4 wv[WL<H>::WPT];
    wload<H>(p, wv);
    tiles<H, 0>(p, sW, wv, sO[threadIdx.x >> 5], sP[threadIdx.x >> 5]);
}

template <int H> __device__ __forceinline__ void fused_kernel(const P& p) {
    __shared__ __align__(16) u32 sW[WL<H>::TOT];
    __shared__ __align__(16) u32 sO[NWARP][64];
    __shared__ __align__(16) u32 sP[NWARP][64];
    uint4 wv[WL<H>::WPT];
    wload<H>(p, wv);
    stamp(p, TID, NTH);
    wstore<H>(sW, wv);
    tiles<H, 1>(p, sW, wv, sO[threadIdx.x >> 5], sP[threadIdx.x >> 5]);
}

extern "C" __global__ void __launch_bounds__(256) k7_stamp(const P p) {
    stamp(p, TID, NTH);
#pragma unroll 1
    for (int j = TID; j < p.nwl; j += NTH) prefetch_l2((const char*)p.wts + 128 * j);   // weights for the mlp kernel
}
extern "C" __global__ void k7_noop(const P p) {}   // diagnostics: launch + sync floor
extern "C" __global__ void __launch_bounds__(NTC) k7_mlp_h16(const P p) { mlp_kernel<16>(p); }
extern "C" __global__ void __launch_bounds__(NTC) k7_mlp_h32(const P p) { mlp_kernel<32>(p); }
extern "C" __global__ void __launch_bounds__(NTC) k7_mlp_h48(const P p) { mlp_kernel<48>(p); }
extern "C" __global__ void __launch_bounds__(NTC) k7_mlp_h64(const P p) { mlp_kernel<64>(p); }
extern "C" __global__ void __launch_bounds__(NTC) k7_fused_h16(const P p) { fused_kernel<16>(p); }
extern "C" __global__ void __launch_bounds__(NTC) k7_fused_h32(const P p) { fused_kernel<32>(p); }
extern "C" __global__ void __launch_bounds__(NTC) k7_fused_h48(const P p) { fused_kernel<48>(p); }
extern "C" __global__ void __launch_bounds__(NTC) k7_fused_h64(const P p) { fused_kernel<64>(p); }
"""

SRC_TAG = hashlib.sha256(CUDA_SRC.encode()).hexdigest()[:12]
ENGINE_H = (16, 32, 48, 64)
KERNELS = ["k7_stamp", "k7_noop"] + [f"k7_{k}_h{h}" for k in ("mlp", "fused") for h in ENGINE_H]


# ----------------------------------------------------------------------------------------------------------------
# model file
# ----------------------------------------------------------------------------------------------------------------
FEATURE_SETS = (["occ", "pos"], ["occ", "pos", "ctx"])


def load_model(path: Path) -> Dict[str, Any]:
    m = torch.load(path, map_location="cpu", weights_only=True)
    if m.get("format") != "k3d-student-v1":
        raise ValueError(f"{path}: unknown format {m.get('format')!r}")
    m["feats"] = list(m["feats"])
    if m["feats"] not in FEATURE_SETS:
        raise ValueError(f"{path}: unsupported features {m['feats']}")
    m["vm_mode"] = os.environ.get("K3D_VM", m.get("vm_mode", "global"))
    if m["vm_mode"] not in ("object", "global", "none"):
        raise ValueError(f"unknown vm mode {m['vm_mode']!r}")
    return m


def effective_last_layer(m: Dict[str, Any]):
    """Last layer weights after folding the 'global' variance matching (identity for 'object' and 'none')."""
    W, b = m["weights"][-1].float(), m["biases"][-1].float()
    if m["vm_mode"] == "global":
        s, mu = m["vm_scale"].float(), m["vm_mean"].float()
        W = W * s[:, None]
        b = b * s + mu * (1 - s)
    return W, b


# ----------------------------------------------------------------------------------------------------------------
# reference implementation (PyTorch)
# ----------------------------------------------------------------------------------------------------------------
_OFF124 = [(dx, dy, dz) for dx in range(-2, 3) for dy in range(-2, 3) for dz in range(-2, 3) if (dx, dy, dz) != (0, 0, 0)]
_OFF125 = [(dx, dy, dz) for dx in range(-2, 3) for dy in range(-2, 3) for dz in range(-2, 3)]


class _Reference:
    def __init__(self, m: Dict[str, Any], device: torch.device):
        self.dev = device
        self.vm_mode = m["vm_mode"]
        self.use_ctx = "ctx" in m["feats"]
        W3, b3 = effective_last_layer(m)
        self.Ws = [w.float().to(device) for w in m["weights"][:-1]] + [W3.to(device)]
        self.bs = [b.float().to(device) for b in m["biases"][:-1]] + [b3.to(device)]
        self.vsc = m["vm_scale"].float().to(device)
        self.off124 = torch.tensor([(dx * GR + dy) * GR + dz for dx, dy, dz in _OFF124], dtype=torch.long, device=device)
        self.off125 = torch.tensor([(dx * 20 + dy) * 20 + dz for dx, dy, dz in _OFF125], dtype=torch.long, device=device)
        # position centre and scale: the bbox centre / half-extent + 1e-6 of a centred object spanning 0..63 (float32,
        # 0-dim tensors so that the division is a true IEEE division on every device, as in training)
        ext = torch.tensor(63.0, dtype=torch.float32, device=device)
        self.pc = torch.full((3,), 31.5, dtype=torch.float32, device=device)
        self.ps = ext / 2 + 1e-6

    def features(self, vox: torch.Tensor) -> torch.Tensor:
        v = vox.to(self.dev).long().clamp(0, 63)
        lin = ((v[:, 0] + 2) * GR + (v[:, 1] + 2)) * GR + (v[:, 2] + 2)
        g = torch.zeros(GR ** 3, dtype=torch.uint8, device=self.dev)
        g[lin] = 1
        occ = g[lin[:, None] + self.off124[None, :]].float()
        p = v.float()
        pos = torch.cat([p / 64.0, (p - self.pc) / self.ps], 1).half().float()
        parts = [occ, pos]
        if self.use_ctx:
            cnt = g.view(GR, GR, GR)[2:66, 2:66, 2:66].reshape(16, 4, 16, 4, 16, 4).sum((1, 3, 5))
            bp = torch.zeros(20, 20, 20, dtype=cnt.dtype, device=self.dev)
            bp[2:18, 2:18, 2:18] = cnt
            q = ((v[:, 0] // 4 + 2) * 20 + (v[:, 1] // 4 + 2)) * 20 + (v[:, 2] // 4 + 2)
            parts.append(bp.view(-1)[q[:, None] + self.off125[None, :]].float() / 64.0)
        return torch.cat(parts, 1)

    def mlp(self, x: torch.Tensor) -> torch.Tensor:
        h = x
        for W, b in zip(self.Ws[:-1], self.bs[:-1]):
            h = torch.nn.functional.silu(h @ W.t() + b)
        return h @ self.Ws[-1].t() + self.bs[-1]

    @torch.no_grad()
    def generate(self, vox: torch.Tensor) -> torch.Tensor:
        if vox.shape[0] == 0:
            return torch.zeros((0, 16), dtype=torch.float32, device=self.dev)
        with torch.autocast(device_type=self.dev.type, enabled=False):
            out = self.mlp(self.features(vox))
            if self.vm_mode == "object":
                mu = out.mean(0, keepdim=True)
                out = mu + (out - mu) * self.vsc
        return out.contiguous()


# ----------------------------------------------------------------------------------------------------------------
# weight packing for the tensor-core kernels (mma.m16n8k16 / m16n8k8 fragment order)
# ----------------------------------------------------------------------------------------------------------------
def _h2(lo: np.ndarray, hi: np.ndarray) -> np.ndarray:
    return lo.astype(np.uint32) | (hi.astype(np.uint32) << 16)


def pack_b16(W: np.ndarray, kpad: int) -> np.ndarray:
    """W [n_out, k] (torch Linear layout, B[k][n] = W[n][k]) -> uint32 [kpad/16][n_out/8][32 lanes][2]."""
    n_out, k = W.shape
    Wp = np.zeros((n_out, kpad), np.float32)
    Wp[:, :k] = W
    Wh = Wp.astype(np.float16).view(np.uint16)
    lane = np.arange(32)
    g, t = lane >> 2, lane & 3
    out = np.zeros((kpad // 16, n_out // 8, 32, 2), np.uint32)
    for kt in range(kpad // 16):
        for nt in range(n_out // 8):
            n, k0 = 8 * nt + g, 16 * kt + 2 * t
            out[kt, nt, :, 0] = _h2(Wh[n, k0], Wh[n, k0 + 1])
            out[kt, nt, :, 1] = _h2(Wh[n, k0 + 8], Wh[n, k0 + 9])
    return out


def pack_b8(W: np.ndarray) -> np.ndarray:
    """W [n_out, k<=8] -> uint32 [n_out/8][32 lanes] (one m16n8k8 B fragment per n-tile)."""
    n_out, k = W.shape
    Wp = np.zeros((n_out, 8), np.float32)
    Wp[:, :k] = W
    Wh = Wp.astype(np.float16).view(np.uint16)
    lane = np.arange(32)
    g, t = lane >> 2, lane & 3
    out = np.zeros((n_out // 8, 32), np.uint32)
    for nt in range(n_out // 8):
        out[nt] = _h2(Wh[8 * nt + g, 2 * t], Wh[8 * nt + g, 2 * t + 1])
    return out


def occ_bit(dx: int, dy: int, dz: int) -> int:
    """Bit (0..123) of neighbour (dx, dy, dz) in the occupancy words the gather of the kernel builds: half-warp 0 packs
    rows (dx, dy) = (-2..-1, -2..2), (0, -2), (0, -1) and the centre row (0, 0) without the centre into bits 0..63,
    half-warp 1 rows (1..2, -2..2), (0, 1), (0, 2) into bits 64..123; 5 bits (dz = -2..2) per row, rows in that order."""
    e = dz + 2
    if dx < 0 or (dx == 0 and dy <= 0):
        base = 0
        i = 5 * (dx + 2) + dy + 2 if dx < 0 else (10 + dy + 2 if dy < 0 else 12)
    else:
        base = 64
        i = 5 * (dx - 1) + dy + 2 if dx > 0 else 10 + dy - 1
    if i == 12:
        assert dz != 0
        return 60 + (e if dz < 0 else e - 1)
    return base + 5 * i + e


def occ_column(b: int) -> int:
    """Layer-1 input column (A-matrix k index) of occupancy bit b. Lane group t reads word t = b >> 5 of a voxel; bit p of
    it goes to pair j = p & 15 (k-tile j >> 1, column half j & 1) as element p >> 4 (see expand in the CUDA source)."""
    t, p = b >> 5, b & 31
    j, e = p & 15, p >> 4
    return 16 * (j >> 1) + 8 * (j & 1) + 2 * t + e


OCC_COLUMNS = np.array([occ_column(occ_bit(*o)) for o in _OFF124], np.int64)   # natural feature -> layer-1 column
assert len(set(OCC_COLUMNS.tolist())) == 124 and OCC_COLUMNS.max() < 128


def _pairs(x: np.ndarray) -> np.ndarray:
    """B fragments [..., NT, 32, 2] -> [..., NT / 2, 32, 4]: (b0, b1) of n-tile 2q, then of n-tile 2q + 1."""
    s = x.shape
    y = x.reshape(s[:-3] + (s[-3] // 2, 2, 32, 2))
    return np.ascontiguousarray(np.moveaxis(y, -3, -2)).reshape(s[:-3] + (s[-3] // 2, 32, 4))


def _bias_quads(b: np.ndarray) -> np.ndarray:
    """Bias [n] -> [n / 8][4 t][4]: (b[m], b[m + 1], b[m], b[m + 1]), m = 8 j + 2 t (C fragment order)."""
    m = 8 * np.arange(b.shape[0] // 8)[:, None] + 2 * np.arange(4)[None, :]
    return np.stack([b[m], b[m + 1], b[m], b[m + 1]], -1).astype(np.float32)


def pack_model(m: Dict[str, Any]) -> Dict[str, np.ndarray]:
    """Fragment-ordered weights of the engine model (features occ + pos, 2 hidden layers): layer-1 occupancy columns
    permuted to the kernel's bit packing, hidden pre-activations halved (SiLU(x) = h + h tanh(h), h = x / 2)."""
    W1, b1 = m["weights"][0].float().numpy(), m["biases"][0].float().numpy()
    W2, b2 = m["weights"][1].float().numpy(), m["biases"][1].float().numpy()
    W3, b3 = (x.numpy() for x in effective_last_layer(m))
    H = W1.shape[0]
    assert W1.shape == (H, 130) and W2.shape == (H, H) and W3.shape == (16, H)
    W1o = np.zeros((H, 128), np.float32)
    W1o[:, OCC_COLUMNS] = W1[:, 0:124]
    return {
        "w1": _pairs(pack_b16(0.5 * W1o, 128)),
        "w1p": np.ascontiguousarray(pack_b8(0.5 * W1[:, 124:130]).reshape(H // 16, 2, 32).transpose(0, 2, 1)),
        "b1": _bias_quads(0.5 * b1),
        "w2": _pairs(pack_b16(0.5 * W2, H)),
        "b2": _bias_quads(0.5 * b2),
        "w3": _pairs(pack_b16(W3, H)).reshape(H // 16, 32, 4),
        "b3": _bias_quads(b3),
    }


def blob_layout(H: int) -> Dict[str, int]:
    """Word offsets of the weight blob (template WL<H> in the CUDA source)."""
    NT, KT, NQ = H // 8, H // 16, H // 16
    L = {"W1": 0}
    L["W1P"] = L["W1"] + 8 * NQ * 128
    L["B1"] = L["W1P"] + NQ * 64
    L["W2"] = L["B1"] + NT * 16
    L["B2"] = L["W2"] + KT * NQ * 128
    L["W3"] = L["B2"] + NT * 16
    L["B3"] = L["W3"] + KT * 128
    L["TOT"] = L["B3"] + 32
    return L


def pack_blob(m: Dict[str, Any]) -> np.ndarray:
    """All engine weights as one uint32 array: W1 occ | W1 pos | b1 | W2 | b2 | W3 | b3 (16-byte aligned sections)."""
    pk = pack_model(m)
    H = int(m["hidden"])
    L = blob_layout(H)
    parts = [pk["w1"], pk["w1p"], pk["b1"].view(np.uint32), pk["w2"], pk["b2"].view(np.uint32), pk["w3"],
             pk["b3"].view(np.uint32)]
    blob = np.concatenate([np.ascontiguousarray(x).reshape(-1).astype(np.uint32) for x in parts])
    assert blob.size == L["TOT"] and L["TOT"] % 4 == 0, (blob.size, L)
    for k in ("W1P", "B1", "W2", "B2", "W3", "B3"):
        assert L[k] % 4 == 0, (k, L)
    return blob


# ----------------------------------------------------------------------------------------------------------------
# CUDA driver API and NVRTC through ctypes
# ----------------------------------------------------------------------------------------------------------------
class _Params(ctypes.Structure):
    """Kernel parameter block (struct P in the CUDA source), passed by value to every kernel."""
    _fields_ = [(k, ctypes.c_void_p) for k in ("vox", "out", "rows", "bar", "wts")] + \
               [(k, ctypes.c_int) for k in ("n", "par", "nwl", "vs0", "vs1", "pad")]


class _Plan(ctypes.Structure):
    """Launch plan for the C shim and the fast-path extension (struct Plan in shim/k3d_shim7.c, shim/k3d_fast7.c)."""
    _fields_ = [("launch", ctypes.c_void_p), ("coop", ctypes.c_void_p), ("fn", ctypes.c_void_p * 3),
                ("params", ctypes.c_void_p), ("n", ctypes.c_void_p), ("mode", ctypes.c_uint), ("grid_cap", ctypes.c_uint),
                ("block", ctypes.c_uint), ("pad", ctypes.c_uint)]


class _Driver:
    def __init__(self):
        self.lib = ctypes.CDLL("libcuda.so.1")
        c_int, c_uint, c_void_p, c_size_t = ctypes.c_int, ctypes.c_uint, ctypes.c_void_p, ctypes.c_size_t
        P = ctypes.POINTER

        def fn(name, args):
            f = getattr(self.lib, name)
            f.argtypes, f.restype = args, c_int
            return f

        self.cuModuleLoadData = fn("cuModuleLoadData", [P(c_void_p), c_void_p])
        self.cuModuleGetFunction = fn("cuModuleGetFunction", [P(c_void_p), c_void_p, ctypes.c_char_p])
        self.cuLaunchKernel = fn("cuLaunchKernel", [c_void_p, c_uint, c_uint, c_uint, c_uint, c_uint, c_uint, c_uint,
                                                    c_void_p, c_void_p, c_void_p])
        self.cuLaunchCooperativeKernel = fn("cuLaunchCooperativeKernel", [c_void_p, c_uint, c_uint, c_uint, c_uint, c_uint,
                                                                          c_uint, c_uint, c_void_p, c_void_p])
        self.cuOccupancy = fn("cuOccupancyMaxActiveBlocksPerMultiprocessor", [P(c_int), c_void_p, c_int, c_size_t])
        self.cuFuncGetAttribute = fn("cuFuncGetAttribute", [P(c_int), c_int, c_void_p])
        self.cuGetErrorString = fn("cuGetErrorString", [c_int, P(ctypes.c_char_p)])
        self.cuCtxGetCurrent = fn("cuCtxGetCurrent", [P(c_void_p)])

    def check(self, r: int, what: str) -> None:
        if r != 0:
            s = ctypes.c_char_p()
            self.cuGetErrorString(r, ctypes.byref(s))
            raise RuntimeError(f"{what}: CUDA error {r} ({(s.value or b'?').decode()})")

    def load(self, image: bytes):
        ctx = ctypes.c_void_p()
        self.check(self.cuCtxGetCurrent(ctypes.byref(ctx)), "cuCtxGetCurrent")
        if not ctx.value:
            raise RuntimeError("no current CUDA context")
        buf = ctypes.create_string_buffer(image, len(image) + 1)  # NUL-terminated (PTX text needs it)
        mod = ctypes.c_void_p()
        self.check(self.cuModuleLoadData(ctypes.byref(mod), buf), "cuModuleLoadData")
        return mod

    def function(self, mod, name: str):
        f = ctypes.c_void_p()
        self.check(self.cuModuleGetFunction(ctypes.byref(f), mod, name.encode()), f"cuModuleGetFunction({name})")
        return f

    def attr(self, f, a: int) -> int:
        v = ctypes.c_int()
        self.check(self.cuFuncGetAttribute(ctypes.byref(v), a, f), "cuFuncGetAttribute")
        return v.value

    def occupancy(self, f, block: int) -> int:
        v = ctypes.c_int()
        self.check(self.cuOccupancy(ctypes.byref(v), f, block, 0), "cuOccupancyMaxActiveBlocksPerMultiprocessor")
        return v.value

    def addr(self, name: str) -> int:
        return ctypes.cast(getattr(self.lib, name), ctypes.c_void_p).value


def _nvrtc_lib():
    cands: List[str] = []
    try:
        import nvidia.cuda_nvrtc as _m  # type: ignore
        for d in _m.__path__:
            cands += sorted(glob.glob(os.path.join(d, "lib", "libnvrtc.so*")))
    except Exception:  # noqa: BLE001
        pass
    for d in [os.path.join(os.path.dirname(torch.__file__), "lib"), os.environ.get("CUDA_HOME", ""), "/usr/local/cuda"]:
        if d:
            cands += sorted(glob.glob(os.path.join(d, "libnvrtc.so*"))) + sorted(glob.glob(os.path.join(d, "lib64", "libnvrtc.so*")))
    cands += ["libnvrtc.so.12", "libnvrtc.so.13", "libnvrtc.so.11.2", "libnvrtc.so"]
    found = ctypes.util.find_library("nvrtc")
    if found:
        cands.append(found)
    errors = []
    for c in cands:
        if "builtins" in os.path.basename(c):
            continue
        try:
            return ctypes.CDLL(c), c
        except OSError as e:
            errors.append(f"{c}: {e}")
    raise RuntimeError("libnvrtc not found: " + "; ".join(errors[-3:]))


def nvrtc_compile(src: str, arch: str) -> bytes:
    """arch 'sm_80' -> cubin, 'compute_75' -> PTX."""
    lib, path = _nvrtc_lib()
    P, c_void_p, c_size_t = ctypes.POINTER, ctypes.c_void_p, ctypes.c_size_t
    prog = c_void_p()

    def ok(r, what):
        if r != 0:
            raise RuntimeError(f"NVRTC {what} failed ({r})")

    ok(lib.nvrtcCreateProgram(ctypes.byref(prog), src.encode(), b"k3d.cu", 0, None, None), "create")
    opts = [f"--gpu-architecture={arch}".encode(), b"--std=c++14", b"-default-device"]
    arr = (ctypes.c_char_p * len(opts))(*opts)
    r = lib.nvrtcCompileProgram(prog, len(opts), arr)
    n = c_size_t()
    lib.nvrtcGetProgramLogSize(prog, ctypes.byref(n))
    logbuf = ctypes.create_string_buffer(n.value + 1)
    lib.nvrtcGetProgramLog(prog, logbuf)
    if r != 0:
        lib.nvrtcDestroyProgram(ctypes.byref(prog))
        raise RuntimeError(f"NVRTC compile ({arch}, {path}) failed: {logbuf.value.decode(errors='replace')[:2000]}")
    if arch.startswith("sm_"):
        ok(lib.nvrtcGetCUBINSize(prog, ctypes.byref(n)), "cubin size")
        out = ctypes.create_string_buffer(n.value)
        ok(lib.nvrtcGetCUBIN(prog, out), "cubin")
    else:
        ok(lib.nvrtcGetPTXSize(prog, ctypes.byref(n)), "ptx size")
        out = ctypes.create_string_buffer(n.value)
        ok(lib.nvrtcGetPTX(prog, out), "ptx")
    lib.nvrtcDestroyProgram(ctypes.byref(prog))
    return out.raw[: n.value]


def _kernel_images(weights_dir: Path, cc):
    """Candidate kernel images in order of preference: shipped cubin, NVRTC cubin, NVRTC PTX (driver JIT)."""
    tag = f"sm_{cc[0]}{cc[1]}"
    f = weights_dir / f"k3d_{SRC_TAG}_{tag}.cubin"
    if f.exists():
        yield f"cubin {f.name}", lambda: f.read_bytes()
    yield f"nvrtc {tag}", lambda: nvrtc_compile(CUDA_SRC, tag)
    yield "nvrtc compute_75 ptx", lambda: nvrtc_compile(CUDA_SRC, "compute_75")


# ----------------------------------------------------------------------------------------------------------------
# CUDA engine
# ----------------------------------------------------------------------------------------------------------------
MODES = ("split", "fused", "coop")
_MODE_ID = {"split": 0, "coop": 1, "fused": 2}
CTA_STAMP = 256    # threads per CTA of the stamp kernel
CTA = 512          # threads per CTA of the mlp / fused kernels (16 warps, 16-voxel tiles spread over the CTAs)


def grid_for(n: int, cap: int) -> int:
    """CTAs of the mlp / fused kernels: one per 16-voxel tile up to the cap (one CTA per SM)."""
    return max(1, min((n + 15) // 16, cap))


class _Engine:
    """CUDA path (features occ + pos, H in ENGINE_H, 2 hidden layers, variance matching folded or none).
    Launch modes: split (stamp + mlp kernels), fused (one kernel with a grid barrier; plain launch, at most one CTA per
    SM) or coop (the same kernel launched cooperatively). Launches go through the C shim (one foreign call per generate)
    or ctypes; the fast path (k3d_fast7) does the same in C."""

    def __init__(self, m: Dict[str, Any], device: torch.device, weights_dir: Path):
        H = int(m["hidden"])
        if m["feats"] != ["occ", "pos"] or H not in ENGINE_H or int(m["layers"]) != 2 or m["vm_mode"] == "object":
            raise ValueError(f"engine supports occ+pos features, H in {ENGINE_H}, 2 hidden layers and folded variance "
                             f"matching; got feats={m['feats']} H={H} L={m['layers']} vm={m['vm_mode']}")
        self.H = H
        self.dev = device
        cc = torch.cuda.get_device_capability(device)
        if cc < (7, 5):
            raise RuntimeError(f"compute capability {cc} < 7.5")
        self.info: Dict[str, Any] = {"cc": f"{cc[0]}.{cc[1]}", "H": H}
        z = dict(device=device)
        torch.zeros(1, **z).add_(1)  # make sure the primary context exists and is current
        torch.cuda.synchronize(device)
        self.cu = _Driver()
        errors = []
        self.mod = None
        with torch.cuda.device(device):
            for name, get in _kernel_images(weights_dir, cc):
                try:
                    t0 = time.perf_counter()
                    image = get()
                    self.mod = self.cu.load(image)
                    self.fn = {k: self.cu.function(self.mod, k) for k in KERNELS}
                    self.info["image"] = name
                    self.info["image_s"] = round(time.perf_counter() - t0, 2)
                    break
                except Exception as e:  # noqa: BLE001
                    errors.append(f"{name}: {e}")
                    self.mod = None
            if self.mod is None:
                raise RuntimeError("no usable kernel image: " + " | ".join(errors))
        if errors:
            self.info["image_errors"] = errors
        self.sms = torch.cuda.get_device_properties(device).multi_processor_count
        self.f_stamp, self.f_mlp, self.f_fused = self.fn["k7_stamp"], self.fn[f"k7_mlp_h{H}"], self.fn[f"k7_fused_h{H}"]
        for k, f in (("stamp", self.f_stamp), ("mlp", self.f_mlp), ("fused", self.f_fused)):
            self.info[f"regs_{k}"] = self.cu.attr(f, 4)   # CU_FUNC_ATTRIBUTE_NUM_REGS
            self.info[f"local_{k}"] = self.cu.attr(f, 3)  # CU_FUNC_ATTRIBUTE_LOCAL_SIZE_BYTES
        self.info["smem_mlp"] = self.cu.attr(self.f_mlp, 1)  # CU_FUNC_ATTRIBUTE_SHARED_SIZE_BYTES
        self.info["occ_mlp"] = self.cu.occupancy(self.f_mlp, CTA)
        occ = self.cu.occupancy(self.f_fused, CTA)
        self.info["occ_fused"] = occ
        self.fused_ok = occ > 0
        self.grid_cap = self.sms  # one CTA per SM: every CTA of the fused kernel is resident (grid barrier)
        if os.environ.get("K3D_GRID_CAP"):
            self.grid_cap = max(1, min(int(os.environ["K3D_GRID_CAP"]), self.sms * max(occ, 1)))
        self.info["grid_cap"] = self.grid_cap
        # one device buffer: occupancy rows of both parities | barrier words (2 x 32 u32) | weight blob
        blob = pack_blob(m)
        self.nstate = 2 * NROW + 32  # int64 words of mutable state
        self.buf = torch.zeros(self.nstate + blob.size // 2, dtype=torch.int64, **z)
        self.buf[self.nstate:].copy_(torch.from_numpy(blob.view(np.int64)))
        base = self.buf.data_ptr()
        p = self.p = _Params()
        p.rows, p.bar, p.wts = base, base + 8 * 2 * NROW, base + 8 * self.nstate
        p.nwl = (blob.size * 4 + 127) // 128
        self.pp = (ctypes.c_void_p * 1)(ctypes.addressof(p))
        p.par = 0  # double-buffer parity of the next call (the parameter block is the one copy every path uses)
        self.launch = self.cu.cuLaunchKernel
        self.coop = self.cu.cuLaunchCooperativeKernel
        self._proto = torch.empty(0, dtype=torch.float32, **z)
        # launch plan, read by the C shim and by the fast-path extension
        pl = self.plan = _Plan()
        pl.launch, pl.coop = self.cu.addr("cuLaunchKernel"), self.cu.addr("cuLaunchCooperativeKernel")
        for i, fnc in enumerate((self.f_stamp, self.f_mlp, self.f_fused)):
            pl.fn[i] = fnc.value
        pl.params = ctypes.addressof(self.pp)
        pl.n = ctypes.addressof(p) + _Params.n.offset
        pl.grid_cap = self.grid_cap
        pl.block = CTA
        self.plan_arg = ctypes.c_void_p(ctypes.addressof(pl))
        self.shim = None
        self._fast_mod = None
        self.mode = "split"
        if os.environ.get("K3D_SHIM", "auto") != "off":
            try:
                self._init_shim(weights_dir)
            except Exception as e:  # noqa: BLE001
                self.info["shim_error"] = f"{type(e).__name__}: {e}"
        self.info["shim"] = self.shim is not None
        self.configure("split")

    def _init_shim(self, weights_dir: Path) -> None:
        import platform
        f = weights_dir / f"k3d_shim7_{platform.machine()}.so"
        if not f.exists():
            raise FileNotFoundError(f.name)
        lib = ctypes.CDLL(str(f))
        run = lib.k3d_run
        run.restype = ctypes.c_int
        self._shim_lib, self.shim = lib, run

    def make_fast(self, weights_dir: Path, slow):
        """generate_impl as one C call (k3d_fast extension): checks, output allocation and the launches of every
        voxel tensor; anything unusual goes to `slow` unchanged."""
        import importlib.machinery
        import importlib.util
        import platform
        f = weights_dir / f"k3d_fast7_{platform.machine()}.abi3.so"
        if not f.exists():
            raise FileNotFoundError(f.name)
        loader = importlib.machinery.ExtensionFileLoader("k3d_fast7", str(f))
        spec = importlib.util.spec_from_file_location("k3d_fast7", str(f), loader=loader)
        mod = importlib.util.module_from_spec(spec)
        loader.exec_module(mod)
        self._fast_mod = mod
        return mod.make(ctypes.addressof(self.plan), ctypes.addressof(self.p), self._proto, torch.int32, slow, 1)

    def modes(self) -> List[str]:
        return ["split", "fused", "coop"] if self.fused_ok else ["split"]

    def configure(self, mode: str) -> None:
        assert mode in MODES, mode
        self.mode = mode
        self.plan.mode = _MODE_ID[mode]

    def name(self) -> str:
        return f"{self.mode}/{'shim' if self.shim is not None else 'ctypes'}"

    def reset_state(self) -> None:
        """Bring the double-buffered scratch state back to the initial (all clear) state."""
        torch.cuda.synchronize(self.dev)
        self.buf[: self.nstate].zero_()
        self.p.par = 0
        torch.cuda.synchronize(self.dev)

    def barrier_timeouts(self) -> int:
        b = self.buf[2 * NROW: self.nstate].view(torch.int32).view(2, 32)
        return int(b[:, 1].sum())

    def _launch(self, n: int) -> int:
        """ctypes launches (the shim does the same in C)."""
        g = grid_for(n, self.grid_cap)
        if self.mode == "split":
            r = self.launch(self.f_stamp, (n + CTA_STAMP - 1) // CTA_STAMP, 1, 1, CTA_STAMP, 1, 1, 0, None, self.pp, None)
            return r | self.launch(self.f_mlp, g, 1, 1, CTA, 1, 1, 0, None, self.pp, None)
        if self.mode == "coop":
            return self.coop(self.f_fused, g, 1, 1, CTA, 1, 1, 0, None, self.pp)
        return self.launch(self.f_fused, g, 1, 1, CTA, 1, 1, 0, None, self.pp, None)

    def generate(self, vox: torch.Tensor) -> torch.Tensor:
        n = vox.shape[0]
        if n == 0:
            return self._proto.new_zeros((0, 16))
        if vox.dtype is not torch.int32 or not vox.is_cuda:
            vox = vox.to(device=self.dev, dtype=torch.int32)
        out = self._proto.new_empty((n, 16))
        p = self.p
        p.vs0, p.vs1 = vox.stride()  # any layout: the harness passes column-major tensors (torch.tensor of an F-ordered npz)
        p.vox = vox.data_ptr()
        p.out = out.data_ptr()
        p.n = n
        r = self.shim(self.plan_arg) if self.shim is not None else self._launch(n)
        if r:
            self.cu.check(r, "kernel launch")
        p.par ^= 1
        return out

    @torch.no_grad()
    def profile(self, vox: torch.Tensor, reps: int = 50) -> Dict[str, float]:
        """Diagnostics only: GPU time of every kernel / mode (CUDA events, median us). A sleep kernel keeps the GPU busy
        while the CPU enqueues, so every interval holds pure kernel time."""
        n = vox.shape[0]
        out = self._proto.new_empty((n, 16))
        p = self.p
        res: Dict[str, List[float]] = {"stamp": [], "mlp": [], "split": [], "fused": [], "coop": []}
        blocker = getattr(torch.cuda, "_sleep", None)
        for _ in range(reps):
            for mode in self.modes():
                p.vox, p.out, p.n = vox.data_ptr(), out.data_ptr(), n
                p.vs0, p.vs1 = vox.stride()
                ev = [torch.cuda.Event(enable_timing=True) for _ in range(3)]
                if blocker is not None:
                    blocker(1000000)
                ev[0].record()
                if mode == "split":
                    self.cu.check(self.launch(self.f_stamp, (n + CTA_STAMP - 1) // CTA_STAMP, 1, 1, CTA_STAMP, 1, 1, 0, None,
                                              self.pp, None), "stamp")
                    ev[1].record()
                    self.cu.check(self.launch(self.f_mlp, grid_for(n, self.grid_cap), 1, 1, CTA, 1, 1, 0, None, self.pp, None),
                                  "mlp")
                    ev[2].record()
                else:
                    keep = self.mode
                    self.mode = mode
                    self.cu.check(self._launch(n), mode)
                    self.mode = keep
                    ev[1].record()
                p.par ^= 1
                torch.cuda.synchronize(self.dev)
                if mode == "split":
                    res["stamp"].append(ev[0].elapsed_time(ev[1]) * 1e3)
                    res["mlp"].append(ev[1].elapsed_time(ev[2]) * 1e3)
                    res["split"].append(ev[0].elapsed_time(ev[2]) * 1e3)
                else:
                    res[mode].append(ev[0].elapsed_time(ev[1]) * 1e3)
        return {k: round(float(np.median(v)), 1) for k, v in res.items() if v}


# ----------------------------------------------------------------------------------------------------------------
# self-check shapes
# ----------------------------------------------------------------------------------------------------------------
def test_shapes(device) -> Dict[str, torch.Tensor]:
    gen = torch.Generator().manual_seed(1234)
    r = torch.arange(64)
    X, Y, Z = torch.meshgrid(r, r, r, indexing="ij")
    P = torch.stack([X, Y, Z], -1).reshape(-1, 3).float()
    d = ((P - 31.5) ** 2).sum(1).sqrt()
    shapes = {
        "shell": P[(d > 24) & (d < 26)],                                           # ~16k, surface-like
        "big": P[(d > 18) & (d < 25)],                                             # ~41k
        "thick": P[(d > 18) & (d < 31)],                                           # ~100k, several waves
        "slab": P[(P[:, 2] >= 30) & (P[:, 2] <= 33) & ((P[:, :2] - 31.5).abs().max(1).values < 20)],
        "random": torch.randint(10, 50, (3000, 3), generator=gen).float(),         # with duplicates
        "cube": P[(P - 20).abs().max(1).values < 10],
        "edges": torch.tensor([[0, 0, 0], [63, 63, 63], [0, 63, 0], [63, 0, 63], [1, 1, 1], [0, 0, 1], [62, 63, 63],
                               [0, 0, 0], [63, 63, 63], [32, 0, 63], [32, 1, 63], [31, 0, 62]]).float(),
        "single": torch.tensor([[5, 40, 63]]).float(),
    }
    out = {}
    for k, v in shapes.items():
        v = v.int()
        perm = torch.randperm(len(v), generator=gen) if k in ("shell", "thick") else torch.arange(len(v))
        out[k] = v[perm].contiguous().to(device)
    for k in ("shell", "big", "edges", "single"):  # column-major copies: the layout the evaluation harness passes
        out[k + "_f"] = out[k].t().contiguous().t()
    return out


def compare(a: torch.Tensor, b: torch.Tensor) -> Dict[str, float]:
    a, b = a.float(), b.float()
    rel = float(((a - b) ** 2).mean() / (b ** 2).mean().clamp(min=1e-12))
    mx = float((a - b).abs().max() / b.abs().max().clamp(min=1e-6))
    return {"rel_mse": rel, "max_rel": mx, "finite": bool(torch.isfinite(a).all())}


# ----------------------------------------------------------------------------------------------------------------
# the generator
# ----------------------------------------------------------------------------------------------------------------
class Solution(LatDitGenerator):
    def __init__(self, model: Dict[str, Any], device: torch.device, weights_dir: Path):
        self.model = model
        self.device = device
        self.weights_dir = weights_dir
        self.ref = _Reference(model, device)
        self.engine: Optional[_Engine] = None
        self.fast = None
        self.info: Dict[str, Any] = {"vm_mode": model["vm_mode"], "hidden": model["hidden"], "src": SRC_TAG}

    @classmethod
    def from_pretrained(cls, pretrained_model_name_or_path, device=None, torch_dtype=None, **kwargs):
        wd = Path(pretrained_model_name_or_path)
        f = wd / "student.pt" if wd.is_dir() else wd
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        device = torch.device(f"cuda:{device}" if isinstance(device, int) else device)
        if device.type == "cuda" and device.index is None:
            device = torch.device("cuda", torch.cuda.current_device())
        self = cls(load_model(f), device, f.parent)
        want = os.environ.get("K3D_ENGINE", "auto")
        if device.type == "cuda" and want != "off":
            self._init_engine(want)
            if self.engine is not None and os.environ.get("K3D_FAST", "auto") != "off":
                self._init_fast()
        self.info["path"] = self.path_name()
        _log(f"ready: H={self.model['hidden']} vm={self.model['vm_mode']} path={self.info['path']}")
        return self

    def path_name(self) -> str:
        if self.engine is None:
            return "torch"
        return "engine/" + self.engine.name() + ("+fast" if self.fast is not None else "")

    def _init_fast(self) -> None:
        """Install the one-call C fast path as this instance's generate_impl, after it reproduces the reference on
        the self-check shapes through the same call the harness makes. Otherwise the Python path stays."""
        eng = self.engine
        try:
            fast = eng.make_fast(self.weights_dir, self._generate_python)
            shapes = test_shapes(self.device)
            order = ["shell_f", "big_f", "edges_f", "single_f", "shell", "thick", "slab", "random", "cube", "edges", "single",
                     "big", "shell_f", "edges_f"]
            eng.reset_state()
            worst = {"rel_mse": 0.0, "max_rel": 0.0, "finite": True}
            with torch.no_grad():
                for k in order:
                    a = fast([shapes[k]], conditioning=None, sampling=None, generator=None)[0]
                    torch.cuda.synchronize(self.device)
                    c = compare(a, self.ref.generate(shapes[k]))
                    worst = {"rel_mse": max(worst["rel_mse"], c["rel_mse"]), "max_rel": max(worst["max_rel"], c["max_rel"]),
                             "finite": worst["finite"] and c["finite"]}
            calls, fallbacks = eng._fast_mod.stats(fast)
            worst.update(calls=calls, fallbacks=fallbacks, timeouts=eng.barrier_timeouts())
            passed = (worst["finite"] and worst["rel_mse"] < 1e-4 and worst["max_rel"] < 0.05 and calls == len(order)
                      and fallbacks == 0 and worst["timeouts"] == 0)
        except Exception as e:  # noqa: BLE001
            worst, passed = f"{type(e).__name__}: {e}", False
        self.info["check fast"] = worst
        _log(f"self-check fast path: {'ok' if passed else 'FAILED'} {worst}")
        eng.reset_state()
        if passed:
            self.fast = fast
            self.generate_impl = fast  # instance attribute: generate() calls straight into C

    # ------------------------------------------------------------------ engine bring-up
    def _init_engine(self, want: str) -> None:
        t0 = time.perf_counter()
        try:
            eng = _Engine(self.model, self.device, self.weights_dir)
        except Exception as e:  # noqa: BLE001
            self.info["engine_error"] = f"{type(e).__name__}: {e}"
            _log(f"engine unavailable, using the PyTorch path: {self.info['engine_error']}")
            return
        self.info.update({f"engine_{k}": v for k, v in eng.info.items()})
        modes = eng.modes()
        if want in MODES:
            modes = [want] if want in modes else []
        shapes = test_shapes(self.device)
        good = []
        for mode in modes:
            eng.configure(mode)
            passed = False
            for attempt in range(2):
                name = eng.name()
                try:
                    eng.reset_state()
                    res = self._self_check(eng, shapes)
                    passed = res["finite"] and res["rel_mse"] < 1e-4 and res["max_rel"] < 0.05 and res["timeouts"] == 0
                except Exception as e:  # noqa: BLE001
                    res, passed = f"{type(e).__name__}: {e}", False
                self.info[f"check {name}"] = res
                _log(f"self-check {name}: {'ok' if passed else 'FAILED'} {res}")
                if passed or eng.shim is None:
                    break
                eng.shim = None  # the launcher shim is suspect: fall back to ctypes launches
            if passed:
                good.append(mode)
        if not good:
            _log("engine disabled (self-check failed), using the PyTorch path")
            return
        if len(good) > 1:
            t = self._bench(eng, shapes, good)
            self.info["bench_us"] = t
            best = min(good, key=lambda md: sum(t["cold"][md]))
            _log(f"engine timings, median us at N={shapes['shell'].shape[0]}/{shapes['big'].shape[0]}: {t}")
        else:
            best = good[0]
        eng.configure(best)
        eng.reset_state()
        self.engine = eng
        self.info["engine_init_s"] = round(time.perf_counter() - t0, 2)

    @torch.no_grad()
    def _self_check(self, eng: _Engine, shapes) -> Dict[str, Any]:
        order = ["shell", "thick", "slab", "random", "cube", "edges", "single", "big", "shell_f", "edges_f", "single_f", "big_f",
                 "shell", "edges", "thick", "random"]
        worst: Dict[str, Any] = {"rel_mse": 0.0, "max_rel": 0.0, "finite": True, "worst": ""}
        for k in order:
            a = eng.generate(shapes[k])
            torch.cuda.synchronize(self.device)
            c = compare(a, self.ref.generate(shapes[k]))
            if c["rel_mse"] >= worst["rel_mse"]:
                worst["worst"] = k
            worst["rel_mse"] = max(worst["rel_mse"], c["rel_mse"])
            worst["max_rel"] = max(worst["max_rel"], c["max_rel"])
            worst["finite"] = worst["finite"] and c["finite"]
        worst["timeouts"] = eng.barrier_timeouts()
        return worst

    @torch.no_grad()
    def _bench(self, eng: _Engine, shapes, modes, rounds: int = 4, reps: int = 8) -> Dict[str, Dict[str, List[float]]]:
        """Warm and cold (L2 and caches flushed by other work before every call, as in the evaluation loop where the
        decoder runs between calls) timing of the launch modes, interleaved to cancel clock drift."""
        dev = self.device
        flush = torch.empty(64 << 18, dtype=torch.float32, device=dev)  # 64 MiB
        fa = torch.randn(512, 512, device=dev)

        def flush_caches():
            flush.add_(1.0)
            torch.mm(fa, fa)

        for md in modes:
            eng.configure(md)
            for _ in range(10):
                eng.generate(shapes["shell_f"])
        torch.cuda.synchronize(dev)
        samples = {(kind, md, k): [] for kind in ("warm", "cold") for md in modes for k in ("shell_f", "big_f")}
        for _ in range(rounds):
            for md in modes:
                eng.configure(md)
                for k in ("shell_f", "big_f"):  # the harness's column-major layout
                    v = shapes[k]
                    for kind in ("warm", "cold"):
                        for _ in range(reps):
                            if kind == "cold":
                                flush_caches()
                            torch.cuda.synchronize(dev)
                            t0 = time.perf_counter()
                            eng.generate(v)
                            torch.cuda.synchronize(dev)
                            samples[(kind, md, k)].append(time.perf_counter() - t0)
        res: Dict[str, Dict[str, List[float]]] = {"warm": {}, "cold": {}}
        for (kind, md, k), ts in samples.items():
            res[kind].setdefault(md, []).append(round(float(np.median(ts)) * 1e6, 1))
        del flush, fa
        return res

    # ------------------------------------------------------------------ inference
    def _generate_python(self, vox_idxs, *, conditioning=None, sampling=None, generator=None, **sampling_kwargs):
        eng = self.engine
        if eng is not None:
            return [eng.generate(v) for v in vox_idxs]
        return [self.ref.generate(v) for v in vox_idxs]

    # the Python path; from_pretrained replaces it on the instance with the C fast path when that passes its check
    generate_impl = _generate_python


def build_cubins(out_dir: str, archs=("sm_75", "sm_80", "sm_86", "sm_89", "sm_90")) -> None:
    """Offline: compile the kernels with NVRTC for the given architectures into out_dir."""
    os.makedirs(out_dir, exist_ok=True)
    for a in archs:
        img = nvrtc_compile(CUDA_SRC, a)
        p = os.path.join(out_dir, f"k3d_{SRC_TAG}_{a}.cubin")
        with open(p, "wb") as f:
            f.write(img)
        print(a, len(img), "bytes ->", p)


if __name__ == "__main__":
    if len(sys.argv) > 2 and sys.argv[1] == "--build-cubins":
        build_cubins(sys.argv[2], tuple(sys.argv[3:]) or ("sm_75", "sm_80", "sm_86", "sm_89", "sm_90"))


In [ ]:
%%writefile /kaggle/working/sub6/solution.py
"""K3D student: a tiny per-voxel network that replaces the lat_dit diffusion model (AIJ 2026, 3D DIT Distillation).

Every voxel's 16-channel latent is predicted by a small MLP from features of the voxel set only:
  occ  124  occupancy of the 5x5x5 neighbourhood (centre excluded), offsets ordered dx, dy, dz = -2..2
  pos    6  voxel / 64 and (voxel - 31.5) / (31.5 + 1e-6), rounded to fp16 (the training features used the bbox centre
            and half of the largest extent; every training and public-test object is centred with half-extent 31.5,
            so the fixed values are identical there)
  (ctx 125  occupied-voxel counts / 64 of the 5x5x5 neighbourhood of 4x4x4 blocks: reference path only)
MLP 130 -> H -> H -> 16 with SiLU, trained to regress the teacher's one-step conditional prediction at t=1 (x0c).
The output deviations are rescaled per channel around the training mean (variance matching folded into the last
layer). All input-dependent computation runs inside generate_impl; from_pretrained only loads and repacks weights.

Two implementations of the same function:
  _Reference  plain PyTorch (any device); fallback and ground truth for the checks;
  _Engine     CUDA kernels launched through the CUDA driver API (no compiler needed at run time), engine v4:
                A  occupancy bit rows (one 64-bit row per (x, y)), atomics; clears the next call's buffers
                C  per-voxel features + MLP on tensor cores (fp16 in, fp32 accumulate); the packed weights are loaded
                   once per CTA with all loads in flight and read from shared memory, activations stay in registers
              launched as two kernels ("split"), or one kernel with a grid barrier (launched cooperatively, "coop",
              or as a plain launch sized to the resident capacity, "fused"). Kernels come from cubins in weights/
              (built from the source below) or are compiled with NVRTC at load. The engine is used only after it
              matches the reference on a self-check at load time; the fastest launch mode is chosen by a benchmark
              with flushed caches.
generate_impl itself is one C call (k3d_fast extension, CPython limited API): tensor checks, output allocation and
the launches, without per-call Python; the Python path stays as its fallback and as the path of the checks.
Environment overrides: K3D_ENGINE = auto | split | coop | fused | off, K3D_SHIM = auto | off, K3D_FAST = auto | off,
K3D_VM = global | none | object, K3D_VERBOSE = 0 | 1.
"""

from __future__ import annotations

import ctypes
import ctypes.util
import glob
import hashlib
import os
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import torch

from kandi3d_image23d import LatDitGenerator

GR = 68                # occupancy grid side with a 2-voxel margin
NROW = GR * GR         # 64-bit occupancy rows (bit z) per (x, y)
NBLK = 4096            # 16^3 coarse 4x4x4 blocks
_VERBOSE = os.environ.get("K3D_VERBOSE", "1") != "0"


def _log(msg: str) -> None:
    if _VERBOSE:
        print(f"[k3d] {msg}", flush=True)


# ----------------------------------------------------------------------------------------------------------------
# CUDA source (no headers needed; compiles with NVRTC or nvcc for sm_75+)
# ----------------------------------------------------------------------------------------------------------------
CUDA_SRC = r"""
// K3D engine v4. Every kernel runs once per call with cold caches, so the design minimises serial memory round trips:
//  - the packed weights (one blob) are loaded with all loads in flight into registers at kernel start and parked in
//    shared memory; the MLP reads B fragments from shared memory only (v3 loaded them from global inside loops);
//  - the 25 occupancy rows of a voxel are loaded at once (one round trip);
//  - layer activations stay in registers (C fragments of one layer are the A fragments of the next);
//  - the position features use the fixed grid centre (every known object is centred with half-extent 31.5), so no
//    bbox reduction.
typedef unsigned int u32;
typedef unsigned long long u64;

#define GR 68
#define NROW 4624
#define FS 7            // u32 per voxel in sF: mask words 0..3, pos half2 4..6 (odd stride: conflict-free)
#define NWARP 8
#define NTC 256         // threads per CTA

struct P {
    const int* vox; float* out; u64* rows; u32* bar; const uint4* wts;
    int n; int par; int nwl; int vs0; int vs1; int pad;   // vox element (i, c) at vox[i * vs0 + c * vs1]
};

// weight blob layout (u32 words): W1 occupancy [8][NT][32][2] | W1 pos [NT][32] | b1 [H] | W2 [KT][NT][32][2] | b2 [H] |
// W3 [KT][2][32][2] | b3 [16]
template <int H> struct WL {
    static constexpr int NT = H / 8, KT = H / 16;
    static constexpr int W1 = 0, W1P = W1 + 512 * NT, B1 = W1P + 32 * NT, W2 = B1 + H, B2 = W2 + 64 * KT * NT,
                         W3 = B2 + H, B3 = W3 + 128 * KT, TOT = B3 + 16, NV = TOT / 4, WPT = (NV + NTC - 1) / NTC;
};

__device__ __forceinline__ int clamp63(int v) { return min(max(v, 0), 63); }

__device__ __forceinline__ void prefetch_l2(const void* a) { asm volatile("prefetch.global.L2 [%0];" :: "l"(a)); }

__device__ __forceinline__ u32 bits2h2(u32 e) {          // 2 bits -> half2 of 0/1
    return ((e & 1u) * 0x3C00u) | (((e >> 1) & 1u) * 0x3C000000u);
}

__device__ __forceinline__ u32 f2h2(float lo, float hi) {  // round-to-nearest fp16 pair
    unsigned short a, b;
    asm("cvt.rn.f16.f32 %0, %1;" : "=h"(a) : "f"(lo));
    asm("cvt.rn.f16.f32 %0, %1;" : "=h"(b) : "f"(hi));
    return (u32)a | ((u32)b << 16);
}

__device__ __forceinline__ float silu(float x) {           // x * sigmoid(x) = h + h * tanh(h), h = x / 2
    const float h = 0.5f * x;
    float t;
    asm("tanh.approx.f32 %0, %1;" : "=f"(t) : "f"(h));
    return fmaf(h, t, h);
}

__device__ __forceinline__ void mma_k16(float* d, const u32* a, u32 b0, u32 b1) {
#if __CUDA_ARCH__ >= 800
    asm("mma.sync.aligned.m16n8k16.row.col.f32.f16.f16.f32 {%0,%1,%2,%3}, {%4,%5,%6,%7}, {%8,%9}, {%0,%1,%2,%3};"
        : "+f"(d[0]), "+f"(d[1]), "+f"(d[2]), "+f"(d[3])
        : "r"(a[0]), "r"(a[1]), "r"(a[2]), "r"(a[3]), "r"(b0), "r"(b1));
#else
    asm("mma.sync.aligned.m16n8k8.row.col.f32.f16.f16.f32 {%0,%1,%2,%3}, {%4,%5}, {%6}, {%0,%1,%2,%3};"
        : "+f"(d[0]), "+f"(d[1]), "+f"(d[2]), "+f"(d[3]) : "r"(a[0]), "r"(a[1]), "r"(b0));
    asm("mma.sync.aligned.m16n8k8.row.col.f32.f16.f16.f32 {%0,%1,%2,%3}, {%4,%5}, {%6}, {%0,%1,%2,%3};"
        : "+f"(d[0]), "+f"(d[1]), "+f"(d[2]), "+f"(d[3]) : "r"(a[2]), "r"(a[3]), "r"(b1));
#endif
}

__device__ __forceinline__ void mma_k8(float* d, u32 a0, u32 a1, u32 b0) {
    asm("mma.sync.aligned.m16n8k8.row.col.f32.f16.f16.f32 {%0,%1,%2,%3}, {%4,%5}, {%6}, {%0,%1,%2,%3};"
        : "+f"(d[0]), "+f"(d[1]), "+f"(d[2]), "+f"(d[3]) : "r"(a0), "r"(a1), "r"(b0));
}

// SiLU of two adjacent n-tiles (C fragments) -> one A fragment (k16) of the next layer
__device__ __forceinline__ void act(u32* a, const float* c0, const float* c1) {
    a[0] = f2h2(silu(c0[0]), silu(c0[1]));
    a[1] = f2h2(silu(c0[2]), silu(c0[3]));
    a[2] = f2h2(silu(c1[0]), silu(c1[1]));
    a[3] = f2h2(silu(c1[2]), silu(c1[3]));
}

// ---- phase A: occupancy rows (bit z of row (x + 2, y + 2)) of this call's parity; clears the other parity
__device__ __forceinline__ void stamp(const P& p, int tid, int nthr) {
    u64* rows = p.rows + (size_t)p.par * NROW;
    uint4* rn = (uint4*)(p.rows + (size_t)(p.par ^ 1) * NROW);
#pragma unroll 1
    for (int j = tid; j < NROW / 16; j += nthr) prefetch_l2(rows + 16 * j);   // rows the gather will read
#pragma unroll 1
    for (int j = tid; j < NROW / 2; j += nthr) rn[j] = make_uint4(0u, 0u, 0u, 0u);
    if (tid == 0) p.bar[(p.par ^ 1) * 32] = 0u;
#pragma unroll 1
    for (int i = tid; i < p.n; i += nthr) {
        const int* q = p.vox + (size_t)i * p.vs0;
        const int x = clamp63(__ldg(q)), y = clamp63(__ldg(q + p.vs1)), z = clamp63(__ldg(q + 2 * p.vs1));
        atomicOr(rows + (x + 2) * GR + (y + 2), 1ull << z);
    }
}

// grid barrier (fused kernel): bar[0] counts arrivals (cleared for the next call by stamp); bar[1] is set if a barrier
// ever times out (never expected; the self-check then rejects the mode)
__device__ __forceinline__ void grid_sync(u32* bar, u32 target) {
    __syncthreads();
    if (threadIdx.x == 0) {
        __threadfence();
        atomicAdd(bar, 1u);
        volatile u32* vb = bar;
        unsigned spins = 0;
        while ((int)(*vb - target) < 0) {
            __nanosleep(32);
            if (++spins > (1u << 22)) { atomicExch(bar + 1, 1u); break; }
        }
        __threadfence();
    }
    __syncthreads();
}

template <int H> __device__ __forceinline__ void wload(const P& p, uint4* v) {
#pragma unroll
    for (int k = 0; k < WL<H>::WPT; k++) {
        const int i = threadIdx.x + k * NTC;
        if (i < WL<H>::NV) v[k] = __ldg(p.wts + i);
    }
}

template <int H> __device__ __forceinline__ void wstore(u32* sW, const uint4* v) {
#pragma unroll
    for (int k = 0; k < WL<H>::WPT; k++) {
        const int i = threadIdx.x + k * NTC;
        if (i < WL<H>::NV) ((uint4*)sW)[i] = v[k];
    }
}

// ---- features of voxel (x, y, z) (lanes < 16 with a voxel): 124 occupancy bits of the 5^3 neighbourhood (one round
// trip: all 25 rows in flight) and 6 position values (x / 64, (x - 31.5) / (31.5 + 1e-6), as fp16; the reciprocal
// product equals the IEEE quotient after fp16 rounding for all 64 coordinates)
#define PRC __int_as_float(0x3d020820)
__device__ __forceinline__ void gather(const u64* rows, u32* sF, int lane, bool on, int x, int y, int z) {
    if (lane < 16) {
        u64 lo = 0ull, hi = 0ull;
        u32 q0 = 0u, q1 = 0u, q2 = 0u;
        if (on) {
            const u64* r0 = rows + x * GR + y;     // row (x + dx + 2, y + dy + 2) = r0[(dx + 2) * GR + dy + 2]
            u64 w[25];
#pragma unroll
            for (int k = 0; k < 25; k++) w[k] = __ldcg(r0 + (k / 5) * GR + (k % 5));
#pragma unroll
            for (int k = 0; k < 25; k++) {
                u32 f = (u32)((z >= 2) ? (w[k] >> (z - 2)) : (w[k] << (2 - z))) & 31u;
                if (k == 12) f = (f & 3u) | ((f >> 1) & 12u);       // centre voxel excluded
                const int st = k <= 12 ? 5 * k : 5 * k - 1;
                if (st < 64) lo |= (u64)f << st; else hi |= (u64)f << (st - 64);
            }
            const float fx = (float)x, fy = (float)y, fz = (float)z;
            q0 = f2h2(fx * 0.015625f, fy * 0.015625f);
            q1 = f2h2(fz * 0.015625f, (fx - 31.5f) * PRC);
            q2 = f2h2((fy - 31.5f) * PRC, (fz - 31.5f) * PRC);
        }
        u32* f = sF + lane * FS;
        f[0] = (u32)lo; f[1] = (u32)(lo >> 32); f[2] = (u32)hi; f[3] = (u32)(hi >> 32);
        f[4] = q0; f[5] = q1; f[6] = q2;
    }
}

// voxel coordinates of lane < 16 of a tile (returns whether the lane has a voxel)
__device__ __forceinline__ bool vload(const P& p, int tile, int lane, int& x, int& y, int& z) {
    const int i = tile * 16 + lane;
    const bool on = lane < 16 && i < p.n;
    if (on) {
        const int* q = p.vox + (size_t)i * p.vs0;
        x = clamp63(__ldg(q)); y = clamp63(__ldg(q + p.vs1)); z = clamp63(__ldg(q + 2 * p.vs1));
    }
    return on;
}

// ---- per-warp tiles of 16 voxels: features -> MLP 130 -> H -> H -> 16 (SiLU) on tensor cores, weights in shared memory.
// Every warp first issues the voxel loads of its first tile, then completes the CTA prologue:
//   MODE 0 (mlp kernel of the split launch): park the weights in shared memory + __syncthreads;
//   MODE 1 (fused kernel): grid barrier (the stamp phase and the weight parking happened before).
template <int H, int MODE>
__device__ __forceinline__ void tiles(const P& p, u32* sW, const uint4* wv, u32* sF, int warp0, int nwarps) {
    typedef WL<H> L;
    constexpr int NT = L::NT, KT = L::KT;
    const int lane = threadIdx.x & 31, g = lane >> 2, t = lane & 3;
    const u64* rows = p.rows + (size_t)p.par * NROW;
    const uint2* w1 = (const uint2*)(sW + L::W1);
    const u32* w1p = sW + L::W1P;
    const uint2* w2 = (const uint2*)(sW + L::W2);
    const uint2* w3 = (const uint2*)(sW + L::W3);
    int tile = warp0;
    int x = 0, y = 0, z = 0;
    bool on = vload(p, tile, lane, x, y, z);
    if (MODE == 0) { wstore<H>(sW, wv); __syncthreads(); }   // prologue outside the loop: wv dies here
    else grid_sync(p.bar + p.par * 32, gridDim.x);
#pragma unroll 1
    while (tile * 16 < p.n) {
        gather(rows, sF, lane, on, x, y, z);
        __syncwarp();
        // layer 1: 8 k-tiles of occupancy bits (A built from the masks) + one k8 tile of position values
        float acc[NT][4];
#pragma unroll
        for (int j = 0; j < NT; j++) {
            const float2 v = *(const float2*)(sW + L::B1 + 8 * j + 2 * t);
            acc[j][0] = v.x; acc[j][1] = v.y; acc[j][2] = v.x; acc[j][3] = v.y;
        }
#pragma unroll 1
        for (int kt = 0; kt < 8; kt++) {
            const int w = kt >> 1, sh = (kt & 1) * 16 + 2 * t;
            const u32 ea = sF[g * FS + w] >> sh, eb = sF[(g + 8) * FS + w] >> sh;
            u32 a[4];
            a[0] = bits2h2(ea); a[1] = bits2h2(eb); a[2] = bits2h2(ea >> 8); a[3] = bits2h2(eb >> 8);
            const uint2* wk = w1 + kt * NT * 32 + lane;
#pragma unroll
            for (int j = 0; j < NT; j++) { const uint2 b = wk[32 * j]; mma_k16(acc[j], a, b.x, b.y); }
        }
        {
            const u32 pa = t < 3 ? sF[g * FS + 4 + t] : 0u, pb = t < 3 ? sF[(g + 8) * FS + 4 + t] : 0u;
#pragma unroll
            for (int j = 0; j < NT; j++) mma_k8(acc[j], pa, pb, w1p[32 * j + lane]);
        }
        u32 a2[KT][4];
#pragma unroll
        for (int k = 0; k < KT; k++) act(a2[k], acc[2 * k], acc[2 * k + 1]);
        // layer 2
        float acc2[NT][4];
#pragma unroll
        for (int j = 0; j < NT; j++) {
            const float2 v = *(const float2*)(sW + L::B2 + 8 * j + 2 * t);
            acc2[j][0] = v.x; acc2[j][1] = v.y; acc2[j][2] = v.x; acc2[j][3] = v.y;
        }
#pragma unroll
        for (int k = 0; k < KT; k++) {
#pragma unroll
            for (int j = 0; j < NT; j++) { const uint2 b = w2[(k * NT + j) * 32 + lane]; mma_k16(acc2[j], a2[k], b.x, b.y); }
        }
        u32 a3[KT][4];
#pragma unroll
        for (int k = 0; k < KT; k++) act(a3[k], acc2[2 * k], acc2[2 * k + 1]);
        // layer 3 (variance matching folded in)
        float o[2][4];
#pragma unroll
        for (int j = 0; j < 2; j++) {
            const float2 v = *(const float2*)(sW + L::B3 + 8 * j + 2 * t);
            o[j][0] = v.x; o[j][1] = v.y; o[j][2] = v.x; o[j][3] = v.y;
        }
#pragma unroll
        for (int k = 0; k < KT; k++) {
#pragma unroll
            for (int j = 0; j < 2; j++) { const uint2 b = w3[(k * 2 + j) * 32 + lane]; mma_k16(o[j], a3[k], b.x, b.y); }
        }
        const int ra = tile * 16 + g, rb = ra + 8;
        if (ra < p.n) {
            *(float2*)(p.out + (size_t)ra * 16 + 2 * t) = make_float2(o[0][0], o[0][1]);
            *(float2*)(p.out + (size_t)ra * 16 + 8 + 2 * t) = make_float2(o[1][0], o[1][1]);
        }
        if (rb < p.n) {
            *(float2*)(p.out + (size_t)rb * 16 + 2 * t) = make_float2(o[0][2], o[0][3]);
            *(float2*)(p.out + (size_t)rb * 16 + 8 + 2 * t) = make_float2(o[1][2], o[1][3]);
        }
        __syncwarp();
        tile += nwarps;
        on = vload(p, tile, lane, x, y, z);
    }
}

#define TID (blockIdx.x * blockDim.x + threadIdx.x)
#define NTH (gridDim.x * blockDim.x)

template <int H> __device__ __forceinline__ void mlp_kernel(const P& p) {
    __shared__ __align__(16) u32 sW[WL<H>::TOT];
    __shared__ u32 sF[NWARP][16 * FS];
    uint4 wv[WL<H>::WPT];
    wload<H>(p, wv);
    tiles<H, 0>(p, sW, wv, sF[threadIdx.x >> 5], TID >> 5, NTH >> 5);
}

template <int H> __device__ __forceinline__ void fused_kernel(const P& p) {
    __shared__ __align__(16) u32 sW[WL<H>::TOT];
    __shared__ u32 sF[NWARP][16 * FS];
    uint4 wv[WL<H>::WPT];
    wload<H>(p, wv);
    stamp(p, TID, NTH);
    wstore<H>(sW, wv);
    tiles<H, 1>(p, sW, wv, sF[threadIdx.x >> 5], TID >> 5, NTH >> 5);
}

extern "C" __global__ void __launch_bounds__(NTC) k4_stamp(const P p) {
    stamp(p, TID, NTH);
#pragma unroll 1
    for (int j = TID; j < p.nwl; j += NTH) prefetch_l2((const char*)p.wts + 128 * j);   // weights for the mlp kernel
}
extern "C" __global__ void k4_noop(const P p) {}   // diagnostics: launch + sync floor
extern "C" __global__ void __launch_bounds__(NTC) k4_mlp_h32(const P p) { mlp_kernel<32>(p); }
extern "C" __global__ void __launch_bounds__(NTC) k4_mlp_h48(const P p) { mlp_kernel<48>(p); }
extern "C" __global__ void __launch_bounds__(NTC) k4_mlp_h64(const P p) { mlp_kernel<64>(p); }
extern "C" __global__ void __launch_bounds__(NTC) k4_fused_h32(const P p) { fused_kernel<32>(p); }
extern "C" __global__ void __launch_bounds__(NTC) k4_fused_h48(const P p) { fused_kernel<48>(p); }
extern "C" __global__ void __launch_bounds__(NTC) k4_fused_h64(const P p) { fused_kernel<64>(p); }
"""

SRC_TAG = hashlib.sha256(CUDA_SRC.encode()).hexdigest()[:12]
ENGINE_H = (32, 48, 64)
KERNELS = ["k4_stamp", "k4_noop"] + [f"k4_{k}_h{h}" for k in ("mlp", "fused") for h in ENGINE_H]


# ----------------------------------------------------------------------------------------------------------------
# model file
# ----------------------------------------------------------------------------------------------------------------
FEATURE_SETS = (["occ", "pos"], ["occ", "pos", "ctx"])


def load_model(path: Path) -> Dict[str, Any]:
    m = torch.load(path, map_location="cpu", weights_only=True)
    if m.get("format") != "k3d-student-v1":
        raise ValueError(f"{path}: unknown format {m.get('format')!r}")
    m["feats"] = list(m["feats"])
    if m["feats"] not in FEATURE_SETS:
        raise ValueError(f"{path}: unsupported features {m['feats']}")
    m["vm_mode"] = os.environ.get("K3D_VM", m.get("vm_mode", "global"))
    if m["vm_mode"] not in ("object", "global", "none"):
        raise ValueError(f"unknown vm mode {m['vm_mode']!r}")
    return m


def effective_last_layer(m: Dict[str, Any]):
    """Last layer weights after folding the 'global' variance matching (identity for 'object' and 'none')."""
    W, b = m["weights"][-1].float(), m["biases"][-1].float()
    if m["vm_mode"] == "global":
        s, mu = m["vm_scale"].float(), m["vm_mean"].float()
        W = W * s[:, None]
        b = b * s + mu * (1 - s)
    return W, b


# ----------------------------------------------------------------------------------------------------------------
# reference implementation (PyTorch)
# ----------------------------------------------------------------------------------------------------------------
_OFF124 = [(dx, dy, dz) for dx in range(-2, 3) for dy in range(-2, 3) for dz in range(-2, 3) if (dx, dy, dz) != (0, 0, 0)]
_OFF125 = [(dx, dy, dz) for dx in range(-2, 3) for dy in range(-2, 3) for dz in range(-2, 3)]


class _Reference:
    def __init__(self, m: Dict[str, Any], device: torch.device):
        self.dev = device
        self.vm_mode = m["vm_mode"]
        self.use_ctx = "ctx" in m["feats"]
        W3, b3 = effective_last_layer(m)
        self.Ws = [w.float().to(device) for w in m["weights"][:-1]] + [W3.to(device)]
        self.bs = [b.float().to(device) for b in m["biases"][:-1]] + [b3.to(device)]
        self.vsc = m["vm_scale"].float().to(device)
        self.off124 = torch.tensor([(dx * GR + dy) * GR + dz for dx, dy, dz in _OFF124], dtype=torch.long, device=device)
        self.off125 = torch.tensor([(dx * 20 + dy) * 20 + dz for dx, dy, dz in _OFF125], dtype=torch.long, device=device)
        # position centre and scale: the bbox centre / half-extent + 1e-6 of a centred object spanning 0..63 (float32,
        # 0-dim tensors so that the division is a true IEEE division on every device, as in training)
        ext = torch.tensor(63.0, dtype=torch.float32, device=device)
        self.pc = torch.full((3,), 31.5, dtype=torch.float32, device=device)
        self.ps = ext / 2 + 1e-6

    def features(self, vox: torch.Tensor) -> torch.Tensor:
        v = vox.to(self.dev).long().clamp(0, 63)
        lin = ((v[:, 0] + 2) * GR + (v[:, 1] + 2)) * GR + (v[:, 2] + 2)
        g = torch.zeros(GR ** 3, dtype=torch.uint8, device=self.dev)
        g[lin] = 1
        occ = g[lin[:, None] + self.off124[None, :]].float()
        p = v.float()
        pos = torch.cat([p / 64.0, (p - self.pc) / self.ps], 1).half().float()
        parts = [occ, pos]
        if self.use_ctx:
            cnt = g.view(GR, GR, GR)[2:66, 2:66, 2:66].reshape(16, 4, 16, 4, 16, 4).sum((1, 3, 5))
            bp = torch.zeros(20, 20, 20, dtype=cnt.dtype, device=self.dev)
            bp[2:18, 2:18, 2:18] = cnt
            q = ((v[:, 0] // 4 + 2) * 20 + (v[:, 1] // 4 + 2)) * 20 + (v[:, 2] // 4 + 2)
            parts.append(bp.view(-1)[q[:, None] + self.off125[None, :]].float() / 64.0)
        return torch.cat(parts, 1)

    def mlp(self, x: torch.Tensor) -> torch.Tensor:
        h = x
        for W, b in zip(self.Ws[:-1], self.bs[:-1]):
            h = torch.nn.functional.silu(h @ W.t() + b)
        return h @ self.Ws[-1].t() + self.bs[-1]

    @torch.no_grad()
    def generate(self, vox: torch.Tensor) -> torch.Tensor:
        if vox.shape[0] == 0:
            return torch.zeros((0, 16), dtype=torch.float32, device=self.dev)
        with torch.autocast(device_type=self.dev.type, enabled=False):
            out = self.mlp(self.features(vox))
            if self.vm_mode == "object":
                mu = out.mean(0, keepdim=True)
                out = mu + (out - mu) * self.vsc
        return out.contiguous()


# ----------------------------------------------------------------------------------------------------------------
# weight packing for the tensor-core kernels (mma.m16n8k16 / m16n8k8 fragment order)
# ----------------------------------------------------------------------------------------------------------------
def _h2(lo: np.ndarray, hi: np.ndarray) -> np.ndarray:
    return lo.astype(np.uint32) | (hi.astype(np.uint32) << 16)


def pack_b16(W: np.ndarray, kpad: int) -> np.ndarray:
    """W [n_out, k] (torch Linear layout, B[k][n] = W[n][k]) -> uint32 [kpad/16][n_out/8][32 lanes][2]."""
    n_out, k = W.shape
    Wp = np.zeros((n_out, kpad), np.float32)
    Wp[:, :k] = W
    Wh = Wp.astype(np.float16).view(np.uint16)
    lane = np.arange(32)
    g, t = lane >> 2, lane & 3
    out = np.zeros((kpad // 16, n_out // 8, 32, 2), np.uint32)
    for kt in range(kpad // 16):
        for nt in range(n_out // 8):
            n, k0 = 8 * nt + g, 16 * kt + 2 * t
            out[kt, nt, :, 0] = _h2(Wh[n, k0], Wh[n, k0 + 1])
            out[kt, nt, :, 1] = _h2(Wh[n, k0 + 8], Wh[n, k0 + 9])
    return out


def pack_b8(W: np.ndarray) -> np.ndarray:
    """W [n_out, k<=8] -> uint32 [n_out/8][32 lanes] (one m16n8k8 B fragment per n-tile)."""
    n_out, k = W.shape
    Wp = np.zeros((n_out, 8), np.float32)
    Wp[:, :k] = W
    Wh = Wp.astype(np.float16).view(np.uint16)
    lane = np.arange(32)
    g, t = lane >> 2, lane & 3
    out = np.zeros((n_out // 8, 32), np.uint32)
    for nt in range(n_out // 8):
        out[nt] = _h2(Wh[8 * nt + g, 2 * t], Wh[8 * nt + g, 2 * t + 1])
    return out


def pack_model(m: Dict[str, Any]) -> Dict[str, np.ndarray]:
    """Fragment-ordered weights of the engine model (features occ + pos, 2 hidden layers)."""
    W1, b1 = m["weights"][0].float().numpy(), m["biases"][0].float().numpy()
    W2, b2 = m["weights"][1].float().numpy(), m["biases"][1].float().numpy()
    W3, b3 = (x.numpy() for x in effective_last_layer(m))
    H = W1.shape[0]
    assert W1.shape == (H, 130) and W2.shape == (H, H) and W3.shape == (16, H)
    return {
        "w1": pack_b16(W1[:, 0:124], 128),
        "w1p": pack_b8(W1[:, 124:130]),
        "b1": b1.astype(np.float32),
        "w2": pack_b16(W2, H),
        "b2": b2.astype(np.float32),
        "w3": pack_b16(W3, H),
        "b3": b3.astype(np.float32),
    }


def blob_layout(H: int) -> Dict[str, int]:
    """Word offsets of the weight blob (template WL<H> in the CUDA source)."""
    NT, KT = H // 8, H // 16
    L = {"W1": 0}
    L["W1P"] = L["W1"] + 512 * NT
    L["B1"] = L["W1P"] + 32 * NT
    L["W2"] = L["B1"] + H
    L["B2"] = L["W2"] + 64 * KT * NT
    L["W3"] = L["B2"] + H
    L["B3"] = L["W3"] + 128 * KT
    L["TOT"] = L["B3"] + 16
    return L


def pack_blob(m: Dict[str, Any]) -> np.ndarray:
    """All engine weights as one uint32 array: W1 occ | W1 pos | b1 | W2 | b2 | W3 | b3 (16-byte aligned sections)."""
    pk = pack_model(m)
    H = int(pk["b1"].shape[0])
    L = blob_layout(H)
    parts = [pk["w1"], pk["w1p"], pk["b1"].view(np.uint32), pk["w2"], pk["b2"].view(np.uint32), pk["w3"],
             pk["b3"].view(np.uint32)]
    blob = np.concatenate([np.ascontiguousarray(x).reshape(-1).astype(np.uint32) for x in parts])
    assert blob.size == L["TOT"] and L["TOT"] % 4 == 0, (blob.size, L)
    return blob


# ----------------------------------------------------------------------------------------------------------------
# CUDA driver API and NVRTC through ctypes
# ----------------------------------------------------------------------------------------------------------------
class _Params(ctypes.Structure):
    """Kernel parameter block (struct P in the CUDA source), passed by value to every kernel."""
    _fields_ = [(k, ctypes.c_void_p) for k in ("vox", "out", "rows", "bar", "wts")] + \
               [(k, ctypes.c_int) for k in ("n", "par", "nwl", "vs0", "vs1", "pad")]


class _Plan(ctypes.Structure):
    """Launch plan for the C shim (struct Plan in shim/k3d_shim4.c)."""
    _fields_ = [("launch", ctypes.c_void_p), ("coop", ctypes.c_void_p), ("fn", ctypes.c_void_p * 3),
                ("params", ctypes.c_void_p), ("n", ctypes.c_void_p), ("mode", ctypes.c_uint), ("grid_fused", ctypes.c_uint)]


class _Driver:
    def __init__(self):
        self.lib = ctypes.CDLL("libcuda.so.1")
        c_int, c_uint, c_void_p, c_size_t = ctypes.c_int, ctypes.c_uint, ctypes.c_void_p, ctypes.c_size_t
        P = ctypes.POINTER

        def fn(name, args):
            f = getattr(self.lib, name)
            f.argtypes, f.restype = args, c_int
            return f

        self.cuModuleLoadData = fn("cuModuleLoadData", [P(c_void_p), c_void_p])
        self.cuModuleGetFunction = fn("cuModuleGetFunction", [P(c_void_p), c_void_p, ctypes.c_char_p])
        self.cuLaunchKernel = fn("cuLaunchKernel", [c_void_p, c_uint, c_uint, c_uint, c_uint, c_uint, c_uint, c_uint,
                                                    c_void_p, c_void_p, c_void_p])
        self.cuLaunchCooperativeKernel = fn("cuLaunchCooperativeKernel", [c_void_p, c_uint, c_uint, c_uint, c_uint, c_uint,
                                                                          c_uint, c_uint, c_void_p, c_void_p])
        self.cuOccupancy = fn("cuOccupancyMaxActiveBlocksPerMultiprocessor", [P(c_int), c_void_p, c_int, c_size_t])
        self.cuFuncGetAttribute = fn("cuFuncGetAttribute", [P(c_int), c_int, c_void_p])
        self.cuGetErrorString = fn("cuGetErrorString", [c_int, P(ctypes.c_char_p)])
        self.cuCtxGetCurrent = fn("cuCtxGetCurrent", [P(c_void_p)])

    def check(self, r: int, what: str) -> None:
        if r != 0:
            s = ctypes.c_char_p()
            self.cuGetErrorString(r, ctypes.byref(s))
            raise RuntimeError(f"{what}: CUDA error {r} ({(s.value or b'?').decode()})")

    def load(self, image: bytes):
        ctx = ctypes.c_void_p()
        self.check(self.cuCtxGetCurrent(ctypes.byref(ctx)), "cuCtxGetCurrent")
        if not ctx.value:
            raise RuntimeError("no current CUDA context")
        buf = ctypes.create_string_buffer(image, len(image) + 1)  # NUL-terminated (PTX text needs it)
        mod = ctypes.c_void_p()
        self.check(self.cuModuleLoadData(ctypes.byref(mod), buf), "cuModuleLoadData")
        return mod

    def function(self, mod, name: str):
        f = ctypes.c_void_p()
        self.check(self.cuModuleGetFunction(ctypes.byref(f), mod, name.encode()), f"cuModuleGetFunction({name})")
        return f

    def attr(self, f, a: int) -> int:
        v = ctypes.c_int()
        self.check(self.cuFuncGetAttribute(ctypes.byref(v), a, f), "cuFuncGetAttribute")
        return v.value

    def occupancy(self, f, block: int) -> int:
        v = ctypes.c_int()
        self.check(self.cuOccupancy(ctypes.byref(v), f, block, 0), "cuOccupancyMaxActiveBlocksPerMultiprocessor")
        return v.value

    def addr(self, name: str) -> int:
        return ctypes.cast(getattr(self.lib, name), ctypes.c_void_p).value


def _nvrtc_lib():
    cands: List[str] = []
    try:
        import nvidia.cuda_nvrtc as _m  # type: ignore
        for d in _m.__path__:
            cands += sorted(glob.glob(os.path.join(d, "lib", "libnvrtc.so*")))
    except Exception:  # noqa: BLE001
        pass
    for d in [os.path.join(os.path.dirname(torch.__file__), "lib"), os.environ.get("CUDA_HOME", ""), "/usr/local/cuda"]:
        if d:
            cands += sorted(glob.glob(os.path.join(d, "libnvrtc.so*"))) + sorted(glob.glob(os.path.join(d, "lib64", "libnvrtc.so*")))
    cands += ["libnvrtc.so.12", "libnvrtc.so.13", "libnvrtc.so.11.2", "libnvrtc.so"]
    found = ctypes.util.find_library("nvrtc")
    if found:
        cands.append(found)
    errors = []
    for c in cands:
        if "builtins" in os.path.basename(c):
            continue
        try:
            return ctypes.CDLL(c), c
        except OSError as e:
            errors.append(f"{c}: {e}")
    raise RuntimeError("libnvrtc not found: " + "; ".join(errors[-3:]))


def nvrtc_compile(src: str, arch: str) -> bytes:
    """arch 'sm_80' -> cubin, 'compute_75' -> PTX."""
    lib, path = _nvrtc_lib()
    P, c_void_p, c_size_t = ctypes.POINTER, ctypes.c_void_p, ctypes.c_size_t
    prog = c_void_p()

    def ok(r, what):
        if r != 0:
            raise RuntimeError(f"NVRTC {what} failed ({r})")

    ok(lib.nvrtcCreateProgram(ctypes.byref(prog), src.encode(), b"k3d.cu", 0, None, None), "create")
    opts = [f"--gpu-architecture={arch}".encode(), b"--std=c++14", b"-default-device"]
    arr = (ctypes.c_char_p * len(opts))(*opts)
    r = lib.nvrtcCompileProgram(prog, len(opts), arr)
    n = c_size_t()
    lib.nvrtcGetProgramLogSize(prog, ctypes.byref(n))
    logbuf = ctypes.create_string_buffer(n.value + 1)
    lib.nvrtcGetProgramLog(prog, logbuf)
    if r != 0:
        lib.nvrtcDestroyProgram(ctypes.byref(prog))
        raise RuntimeError(f"NVRTC compile ({arch}, {path}) failed: {logbuf.value.decode(errors='replace')[:2000]}")
    if arch.startswith("sm_"):
        ok(lib.nvrtcGetCUBINSize(prog, ctypes.byref(n)), "cubin size")
        out = ctypes.create_string_buffer(n.value)
        ok(lib.nvrtcGetCUBIN(prog, out), "cubin")
    else:
        ok(lib.nvrtcGetPTXSize(prog, ctypes.byref(n)), "ptx size")
        out = ctypes.create_string_buffer(n.value)
        ok(lib.nvrtcGetPTX(prog, out), "ptx")
    lib.nvrtcDestroyProgram(ctypes.byref(prog))
    return out.raw[: n.value]


def _kernel_images(weights_dir: Path, cc):
    """Candidate kernel images in order of preference: shipped cubin, NVRTC cubin, NVRTC PTX (driver JIT)."""
    tag = f"sm_{cc[0]}{cc[1]}"
    f = weights_dir / f"k3d_{SRC_TAG}_{tag}.cubin"
    if f.exists():
        yield f"cubin {f.name}", lambda: f.read_bytes()
    yield f"nvrtc {tag}", lambda: nvrtc_compile(CUDA_SRC, tag)
    yield "nvrtc compute_75 ptx", lambda: nvrtc_compile(CUDA_SRC, "compute_75")


# ----------------------------------------------------------------------------------------------------------------
# CUDA engine
# ----------------------------------------------------------------------------------------------------------------
MODES = ("split", "fused", "coop")
_MODE_ID = {"split": 0, "coop": 1, "fused": 2}
CTA = 256          # threads per CTA of every kernel
VPC = 128          # voxels per CTA of the mlp / fused kernels (8 warps x 16)


class _Engine:
    """CUDA path (features occ + pos, H in ENGINE_H, 2 hidden layers, variance matching folded or none).
    Launch modes: split (stamp + mlp kernels), fused (one kernel with a grid barrier; plain launch with the grid capped at
    the resident capacity) or coop (the same kernel launched cooperatively). Launches go through the C shim (one foreign
    call per generate) or ctypes."""

    def __init__(self, m: Dict[str, Any], device: torch.device, weights_dir: Path):
        H = int(m["hidden"])
        if m["feats"] != ["occ", "pos"] or H not in ENGINE_H or int(m["layers"]) != 2 or m["vm_mode"] == "object":
            raise ValueError(f"engine supports occ+pos features, H in {ENGINE_H}, 2 hidden layers and folded variance "
                             f"matching; got feats={m['feats']} H={H} L={m['layers']} vm={m['vm_mode']}")
        self.H = H
        self.dev = device
        cc = torch.cuda.get_device_capability(device)
        if cc < (7, 5):
            raise RuntimeError(f"compute capability {cc} < 7.5")
        self.info: Dict[str, Any] = {"cc": f"{cc[0]}.{cc[1]}", "H": H}
        z = dict(device=device)
        torch.zeros(1, **z).add_(1)  # make sure the primary context exists and is current
        torch.cuda.synchronize(device)
        self.cu = _Driver()
        errors = []
        self.mod = None
        with torch.cuda.device(device):
            for name, get in _kernel_images(weights_dir, cc):
                try:
                    t0 = time.perf_counter()
                    image = get()
                    self.mod = self.cu.load(image)
                    self.fn = {k: self.cu.function(self.mod, k) for k in KERNELS}
                    self.info["image"] = name
                    self.info["image_s"] = round(time.perf_counter() - t0, 2)
                    break
                except Exception as e:  # noqa: BLE001
                    errors.append(f"{name}: {e}")
                    self.mod = None
            if self.mod is None:
                raise RuntimeError("no usable kernel image: " + " | ".join(errors))
        if errors:
            self.info["image_errors"] = errors
        self.sms = torch.cuda.get_device_properties(device).multi_processor_count
        self.f_stamp, self.f_mlp, self.f_fused = self.fn["k4_stamp"], self.fn[f"k4_mlp_h{H}"], self.fn[f"k4_fused_h{H}"]
        for k, f in (("stamp", self.f_stamp), ("mlp", self.f_mlp), ("fused", self.f_fused)):
            self.info[f"regs_{k}"] = self.cu.attr(f, 4)   # CU_FUNC_ATTRIBUTE_NUM_REGS
            self.info[f"local_{k}"] = self.cu.attr(f, 3)  # CU_FUNC_ATTRIBUTE_LOCAL_SIZE_BYTES
        self.info["smem_mlp"] = self.cu.attr(self.f_mlp, 1)  # CU_FUNC_ATTRIBUTE_SHARED_SIZE_BYTES
        self.info["occ_mlp"] = self.cu.occupancy(self.f_mlp, CTA)
        occ = self.cu.occupancy(self.f_fused, CTA)
        self.info["occ_fused"] = occ
        self.fused_ok = occ > 0
        self.fused_max = self.sms * max(occ, 1)
        # one device buffer: occupancy rows of both parities | barrier words (2 x 32 u32) | weight blob
        blob = pack_blob(m)
        self.nstate = 2 * NROW + 32  # int64 words of mutable state
        self.buf = torch.zeros(self.nstate + blob.size // 2, dtype=torch.int64, **z)
        self.buf[self.nstate:].copy_(torch.from_numpy(blob.view(np.int64)))
        base = self.buf.data_ptr()
        p = self.p = _Params()
        p.rows, p.bar, p.wts = base, base + 8 * 2 * NROW, base + 8 * self.nstate
        p.nwl = (blob.size * 4 + 127) // 128
        self.pp = (ctypes.c_void_p * 1)(ctypes.addressof(p))
        p.par = 0  # double-buffer parity of the next call (the parameter block is the one copy every path uses)
        self.launch = self.cu.cuLaunchKernel
        self.coop = self.cu.cuLaunchCooperativeKernel
        self._proto = torch.empty(0, dtype=torch.float32, **z)
        # launch plan, read by the C shim and by the fast-path extension
        pl = self.plan = _Plan()
        pl.launch, pl.coop = self.cu.addr("cuLaunchKernel"), self.cu.addr("cuLaunchCooperativeKernel")
        for i, fnc in enumerate((self.f_stamp, self.f_mlp, self.f_fused)):
            pl.fn[i] = fnc.value
        pl.params = ctypes.addressof(self.pp)
        pl.n = ctypes.addressof(p) + _Params.n.offset
        pl.grid_fused = self.fused_max
        self.plan_arg = ctypes.c_void_p(ctypes.addressof(pl))
        self.shim = None
        self._fast_mod = None
        self.mode = "split"
        if os.environ.get("K3D_SHIM", "auto") != "off":
            try:
                self._init_shim(weights_dir)
            except Exception as e:  # noqa: BLE001
                self.info["shim_error"] = f"{type(e).__name__}: {e}"
        self.info["shim"] = self.shim is not None
        self.configure("split")

    def _init_shim(self, weights_dir: Path) -> None:
        import platform
        f = weights_dir / f"k3d_shim4_{platform.machine()}.so"
        if not f.exists():
            raise FileNotFoundError(f.name)
        lib = ctypes.CDLL(str(f))
        run = lib.k3d_run
        run.restype = ctypes.c_int
        self._shim_lib, self.shim = lib, run

    def make_fast(self, weights_dir: Path, slow):
        """generate_impl as one C call (k3d_fast extension): checks, output allocation and the launches of every
        voxel tensor; anything unusual goes to `slow` unchanged."""
        import importlib.machinery
        import importlib.util
        import platform
        f = weights_dir / f"k3d_fast_{platform.machine()}.abi3.so"
        if not f.exists():
            raise FileNotFoundError(f.name)
        loader = importlib.machinery.ExtensionFileLoader("k3d_fast", str(f))
        spec = importlib.util.spec_from_file_location("k3d_fast", str(f), loader=loader)
        mod = importlib.util.module_from_spec(spec)
        loader.exec_module(mod)
        self._fast_mod = mod
        return mod.make(ctypes.addressof(self.plan), ctypes.addressof(self.p), self._proto, torch.int32, slow, 1)

    def modes(self) -> List[str]:
        return ["split", "fused", "coop"] if self.fused_ok else ["split"]

    def configure(self, mode: str) -> None:
        assert mode in MODES, mode
        self.mode = mode
        self.plan.mode = _MODE_ID[mode]

    def name(self) -> str:
        return f"{self.mode}/{'shim' if self.shim is not None else 'ctypes'}"

    def reset_state(self) -> None:
        """Bring the double-buffered scratch state back to the initial (all clear) state."""
        torch.cuda.synchronize(self.dev)
        self.buf[: self.nstate].zero_()
        self.p.par = 0
        torch.cuda.synchronize(self.dev)

    def barrier_timeouts(self) -> int:
        b = self.buf[2 * NROW: self.nstate].view(torch.int32).view(2, 32)
        return int(b[:, 1].sum())

    def fused_grid(self, n: int) -> int:
        return max(1, min((n + VPC - 1) // VPC, self.fused_max))

    def _launch(self, n: int) -> int:
        """ctypes launches (the shim does the same in C)."""
        if self.mode == "split":
            r = self.launch(self.f_stamp, (n + CTA - 1) // CTA, 1, 1, CTA, 1, 1, 0, None, self.pp, None)
            return r | self.launch(self.f_mlp, (n + VPC - 1) // VPC, 1, 1, CTA, 1, 1, 0, None, self.pp, None)
        if self.mode == "coop":
            return self.coop(self.f_fused, self.fused_grid(n), 1, 1, CTA, 1, 1, 0, None, self.pp)
        return self.launch(self.f_fused, self.fused_grid(n), 1, 1, CTA, 1, 1, 0, None, self.pp, None)

    def generate(self, vox: torch.Tensor) -> torch.Tensor:
        n = vox.shape[0]
        if n == 0:
            return self._proto.new_zeros((0, 16))
        if vox.dtype is not torch.int32 or not vox.is_cuda:
            vox = vox.to(device=self.dev, dtype=torch.int32)
        out = self._proto.new_empty((n, 16))
        p = self.p
        p.vs0, p.vs1 = vox.stride()  # any layout: the harness passes column-major tensors (torch.tensor of an F-ordered npz)
        p.vox = vox.data_ptr()
        p.out = out.data_ptr()
        p.n = n
        r = self.shim(self.plan_arg) if self.shim is not None else self._launch(n)
        if r:
            self.cu.check(r, "kernel launch")
        p.par ^= 1
        return out

    @torch.no_grad()
    def profile(self, vox: torch.Tensor, reps: int = 50) -> Dict[str, float]:
        """Diagnostics only: GPU time of every kernel / mode (CUDA events, median us). A sleep kernel keeps the GPU busy
        while the CPU enqueues, so every interval holds pure kernel time."""
        n = vox.shape[0]
        out = self._proto.new_empty((n, 16))
        p = self.p
        res: Dict[str, List[float]] = {"stamp": [], "mlp": [], "split": [], "fused": [], "coop": []}
        blocker = getattr(torch.cuda, "_sleep", None)
        for _ in range(reps):
            for mode in self.modes():
                p.vox, p.out, p.n = vox.data_ptr(), out.data_ptr(), n
                p.vs0, p.vs1 = vox.stride()
                ev = [torch.cuda.Event(enable_timing=True) for _ in range(3)]
                if blocker is not None:
                    blocker(1000000)
                ev[0].record()
                if mode == "split":
                    self.cu.check(self.launch(self.f_stamp, (n + CTA - 1) // CTA, 1, 1, CTA, 1, 1, 0, None, self.pp, None), "stamp")
                    ev[1].record()
                    self.cu.check(self.launch(self.f_mlp, (n + VPC - 1) // VPC, 1, 1, CTA, 1, 1, 0, None, self.pp, None), "mlp")
                    ev[2].record()
                else:
                    keep = self.mode
                    self.mode = mode
                    self.cu.check(self._launch(n), mode)
                    self.mode = keep
                    ev[1].record()
                p.par ^= 1
                torch.cuda.synchronize(self.dev)
                if mode == "split":
                    res["stamp"].append(ev[0].elapsed_time(ev[1]) * 1e3)
                    res["mlp"].append(ev[1].elapsed_time(ev[2]) * 1e3)
                    res["split"].append(ev[0].elapsed_time(ev[2]) * 1e3)
                else:
                    res[mode].append(ev[0].elapsed_time(ev[1]) * 1e3)
        return {k: round(float(np.median(v)), 1) for k, v in res.items() if v}


# ----------------------------------------------------------------------------------------------------------------
# self-check shapes
# ----------------------------------------------------------------------------------------------------------------
def test_shapes(device) -> Dict[str, torch.Tensor]:
    gen = torch.Generator().manual_seed(1234)
    r = torch.arange(64)
    X, Y, Z = torch.meshgrid(r, r, r, indexing="ij")
    P = torch.stack([X, Y, Z], -1).reshape(-1, 3).float()
    d = ((P - 31.5) ** 2).sum(1).sqrt()
    shapes = {
        "shell": P[(d > 24) & (d < 26)],                                           # ~16k, surface-like
        "big": P[(d > 18) & (d < 25)],                                             # ~41k
        "thick": P[(d > 18) & (d < 31)],                                           # ~100k, several waves
        "slab": P[(P[:, 2] >= 30) & (P[:, 2] <= 33) & ((P[:, :2] - 31.5).abs().max(1).values < 20)],
        "random": torch.randint(10, 50, (3000, 3), generator=gen).float(),         # with duplicates
        "cube": P[(P - 20).abs().max(1).values < 10],
        "edges": torch.tensor([[0, 0, 0], [63, 63, 63], [0, 63, 0], [63, 0, 63], [1, 1, 1], [0, 0, 1], [62, 63, 63],
                               [0, 0, 0], [63, 63, 63], [32, 0, 63], [32, 1, 63], [31, 0, 62]]).float(),
        "single": torch.tensor([[5, 40, 63]]).float(),
    }
    out = {}
    for k, v in shapes.items():
        v = v.int()
        perm = torch.randperm(len(v), generator=gen) if k in ("shell", "thick") else torch.arange(len(v))
        out[k] = v[perm].contiguous().to(device)
    for k in ("shell", "big", "edges", "single"):  # column-major copies: the layout the evaluation harness passes
        out[k + "_f"] = out[k].t().contiguous().t()
    return out


def compare(a: torch.Tensor, b: torch.Tensor) -> Dict[str, float]:
    a, b = a.float(), b.float()
    rel = float(((a - b) ** 2).mean() / (b ** 2).mean().clamp(min=1e-12))
    mx = float((a - b).abs().max() / b.abs().max().clamp(min=1e-6))
    return {"rel_mse": rel, "max_rel": mx, "finite": bool(torch.isfinite(a).all())}


# ----------------------------------------------------------------------------------------------------------------
# the generator
# ----------------------------------------------------------------------------------------------------------------
class Solution(LatDitGenerator):
    def __init__(self, model: Dict[str, Any], device: torch.device, weights_dir: Path):
        self.model = model
        self.device = device
        self.weights_dir = weights_dir
        self.ref = _Reference(model, device)
        self.engine: Optional[_Engine] = None
        self.fast = None
        self.info: Dict[str, Any] = {"vm_mode": model["vm_mode"], "hidden": model["hidden"], "src": SRC_TAG}

    @classmethod
    def from_pretrained(cls, pretrained_model_name_or_path, device=None, torch_dtype=None, **kwargs):
        wd = Path(pretrained_model_name_or_path)
        f = wd / "student.pt" if wd.is_dir() else wd
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        device = torch.device(f"cuda:{device}" if isinstance(device, int) else device)
        if device.type == "cuda" and device.index is None:
            device = torch.device("cuda", torch.cuda.current_device())
        self = cls(load_model(f), device, f.parent)
        want = os.environ.get("K3D_ENGINE", "auto")
        if device.type == "cuda" and want != "off":
            self._init_engine(want)
            if self.engine is not None and os.environ.get("K3D_FAST", "auto") != "off":
                self._init_fast()
        self.info["path"] = self.path_name()
        _log(f"ready: H={self.model['hidden']} vm={self.model['vm_mode']} path={self.info['path']}")
        return self

    def path_name(self) -> str:
        if self.engine is None:
            return "torch"
        return "engine/" + self.engine.name() + ("+fast" if self.fast is not None else "")

    def _init_fast(self) -> None:
        """Install the one-call C fast path as this instance's generate_impl, after it reproduces the reference on
        the self-check shapes through the same call the harness makes. Otherwise the Python path stays."""
        eng = self.engine
        try:
            fast = eng.make_fast(self.weights_dir, self._generate_python)
            shapes = test_shapes(self.device)
            order = ["shell_f", "big_f", "edges_f", "single_f", "shell", "thick", "slab", "random", "cube", "edges", "single",
                     "big", "shell_f", "edges_f"]
            eng.reset_state()
            worst = {"rel_mse": 0.0, "max_rel": 0.0, "finite": True}
            with torch.no_grad():
                for k in order:
                    a = fast([shapes[k]], conditioning=None, sampling=None, generator=None)[0]
                    torch.cuda.synchronize(self.device)
                    c = compare(a, self.ref.generate(shapes[k]))
                    worst = {"rel_mse": max(worst["rel_mse"], c["rel_mse"]), "max_rel": max(worst["max_rel"], c["max_rel"]),
                             "finite": worst["finite"] and c["finite"]}
            calls, fallbacks = eng._fast_mod.stats(fast)
            worst.update(calls=calls, fallbacks=fallbacks, timeouts=eng.barrier_timeouts())
            passed = (worst["finite"] and worst["rel_mse"] < 1e-4 and worst["max_rel"] < 0.05 and calls == len(order)
                      and fallbacks == 0 and worst["timeouts"] == 0)
        except Exception as e:  # noqa: BLE001
            worst, passed = f"{type(e).__name__}: {e}", False
        self.info["check fast"] = worst
        _log(f"self-check fast path: {'ok' if passed else 'FAILED'} {worst}")
        eng.reset_state()
        if passed:
            self.fast = fast
            self.generate_impl = fast  # instance attribute: generate() calls straight into C

    # ------------------------------------------------------------------ engine bring-up
    def _init_engine(self, want: str) -> None:
        t0 = time.perf_counter()
        try:
            eng = _Engine(self.model, self.device, self.weights_dir)
        except Exception as e:  # noqa: BLE001
            self.info["engine_error"] = f"{type(e).__name__}: {e}"
            _log(f"engine unavailable, using the PyTorch path: {self.info['engine_error']}")
            return
        self.info.update({f"engine_{k}": v for k, v in eng.info.items()})
        modes = eng.modes()
        if want in MODES:
            modes = [want] if want in modes else []
        shapes = test_shapes(self.device)
        good = []
        for mode in modes:
            eng.configure(mode)
            passed = False
            for attempt in range(2):
                name = eng.name()
                try:
                    eng.reset_state()
                    res = self._self_check(eng, shapes)
                    passed = res["finite"] and res["rel_mse"] < 1e-4 and res["max_rel"] < 0.05 and res["timeouts"] == 0
                except Exception as e:  # noqa: BLE001
                    res, passed = f"{type(e).__name__}: {e}", False
                self.info[f"check {name}"] = res
                _log(f"self-check {name}: {'ok' if passed else 'FAILED'} {res}")
                if passed or eng.shim is None:
                    break
                eng.shim = None  # the launcher shim is suspect: fall back to ctypes launches
            if passed:
                good.append(mode)
        if not good:
            _log("engine disabled (self-check failed), using the PyTorch path")
            return
        if len(good) > 1:
            t = self._bench(eng, shapes, good)
            self.info["bench_us"] = t
            best = min(good, key=lambda md: sum(t["cold"][md]))
            _log(f"engine timings, median us at N={shapes['shell'].shape[0]}/{shapes['big'].shape[0]}: {t}")
        else:
            best = good[0]
        eng.configure(best)
        eng.reset_state()
        self.engine = eng
        self.info["engine_init_s"] = round(time.perf_counter() - t0, 2)

    @torch.no_grad()
    def _self_check(self, eng: _Engine, shapes) -> Dict[str, Any]:
        order = ["shell", "thick", "slab", "random", "cube", "edges", "single", "big", "shell_f", "edges_f", "single_f", "big_f",
                 "shell", "edges", "thick", "random"]
        worst: Dict[str, Any] = {"rel_mse": 0.0, "max_rel": 0.0, "finite": True, "worst": ""}
        for k in order:
            a = eng.generate(shapes[k])
            torch.cuda.synchronize(self.device)
            c = compare(a, self.ref.generate(shapes[k]))
            if c["rel_mse"] >= worst["rel_mse"]:
                worst["worst"] = k
            worst["rel_mse"] = max(worst["rel_mse"], c["rel_mse"])
            worst["max_rel"] = max(worst["max_rel"], c["max_rel"])
            worst["finite"] = worst["finite"] and c["finite"]
        worst["timeouts"] = eng.barrier_timeouts()
        return worst

    @torch.no_grad()
    def _bench(self, eng: _Engine, shapes, modes, rounds: int = 4, reps: int = 8) -> Dict[str, Dict[str, List[float]]]:
        """Warm and cold (L2 and caches flushed by other work before every call, as in the evaluation loop where the
        decoder runs between calls) timing of the launch modes, interleaved to cancel clock drift."""
        dev = self.device
        flush = torch.empty(64 << 18, dtype=torch.float32, device=dev)  # 64 MiB
        fa = torch.randn(512, 512, device=dev)

        def flush_caches():
            flush.add_(1.0)
            torch.mm(fa, fa)

        for md in modes:
            eng.configure(md)
            for _ in range(10):
                eng.generate(shapes["shell_f"])
        torch.cuda.synchronize(dev)
        samples = {(kind, md, k): [] for kind in ("warm", "cold") for md in modes for k in ("shell_f", "big_f")}
        for _ in range(rounds):
            for md in modes:
                eng.configure(md)
                for k in ("shell_f", "big_f"):  # the harness's column-major layout
                    v = shapes[k]
                    for kind in ("warm", "cold"):
                        for _ in range(reps):
                            if kind == "cold":
                                flush_caches()
                            torch.cuda.synchronize(dev)
                            t0 = time.perf_counter()
                            eng.generate(v)
                            torch.cuda.synchronize(dev)
                            samples[(kind, md, k)].append(time.perf_counter() - t0)
        res: Dict[str, Dict[str, List[float]]] = {"warm": {}, "cold": {}}
        for (kind, md, k), ts in samples.items():
            res[kind].setdefault(md, []).append(round(float(np.median(ts)) * 1e6, 1))
        del flush, fa
        return res

    # ------------------------------------------------------------------ inference
    def _generate_python(self, vox_idxs, *, conditioning=None, sampling=None, generator=None, **sampling_kwargs):
        eng = self.engine
        if eng is not None:
            return [eng.generate(v) for v in vox_idxs]
        return [self.ref.generate(v) for v in vox_idxs]

    # the Python path; from_pretrained replaces it on the instance with the C fast path when that passes its check
    generate_impl = _generate_python


def build_cubins(out_dir: str, archs=("sm_75", "sm_80", "sm_86", "sm_89", "sm_90")) -> None:
    """Offline: compile the kernels with NVRTC for the given architectures into out_dir."""
    os.makedirs(out_dir, exist_ok=True)
    for a in archs:
        img = nvrtc_compile(CUDA_SRC, a)
        p = os.path.join(out_dir, f"k3d_{SRC_TAG}_{a}.cubin")
        with open(p, "wb") as f:
            f.write(img)
        print(a, len(img), "bytes ->", p)


if __name__ == "__main__":
    if len(sys.argv) > 2 and sys.argv[1] == "--build-cubins":
        build_cubins(sys.argv[2], tuple(sys.argv[3:]) or ("sm_75", "sm_80", "sm_86", "sm_89", "sm_90"))


In [ ]:
%%writefile /kaggle/working/bench11.py
"""K3D-11 benchmark: output tensors through DLPack slots (engine v9) in the regime of the official evaluation loop.

K3D-10 split a cold call (GPU idle for a second, CPU caches flushed by other work) of engine v7 H16 on T4: input checks
6 us, proto.new_empty 30 us, the first kernel launch 19 us, then the harness's synchronize. Engine v9 takes the output
from a pool of device slots wrapped with DLPack (torch._C._from_dlpack: no dispatcher, no caching allocator) and can
launch the kernel before the tensor object exists (the slot pointer is known), so the GPU works while the CPU builds it.

Variants (k3d_fast9 driving engine v7 kernels, rdtsc stage stamps): C path without launches with new_empty / DLPack;
empty kernel with new_empty / DLPack + launch first; v7 H16 with new_empty / DLPack / DLPack + launch first; the
production solutions v7 H16, v6 H64 (leaderboard) and the v9 H16 zip exactly as packed. Regimes: warm (back-to-back),
loop (decoder-like GPU burst, 1 s of CPU work with the GPU idle, H2D as run_lat_dit_eval.py, call), idle (1 s sleep,
H2D, call). Writes <WORK>/bench/bench11_<tag>.json."""

import ctypes
import importlib.machinery
import importlib.util
import json
import os
import sys
import time
import zlib
from pathlib import Path

import numpy as np
import torch

WORK = Path(os.environ.get("K3D_WORK", "/kaggle/working"))
VOXDIR = Path(os.environ.get("K3D_VOXDIR", "/tmp/k3d_vox"))
IDLE_S = float(os.environ.get("K3D_IDLE_S", "1.0"))
REPS = int(os.environ.get("K3D_REPS", "3"))
if str(WORK / "stub") not in sys.path:
    sys.path.insert(0, str(WORK / "stub"))
from kandi3d_image23d.lat_dit_protocol import LatDitGenerator  # noqa: E402
from kandi3d_image23d.runtime import LatDitConditioning  # noqa: E402

F_WAKE, F_SPLIT_EARLY, F_TRACE, F_DLPACK, F_LAUNCH_FIRST = 1, 2, 4, 8, 16
STAGES = ["wake", "checks", "vox", "out", "launch", "exit"]


class Plan8(ctypes.Structure):
    """struct Plan of shim/k3d_fast9.c"""
    _fields_ = [("launch", ctypes.c_void_p), ("coop", ctypes.c_void_p), ("fn", ctypes.c_void_p * 3),
                ("params", ctypes.c_void_p), ("n", ctypes.c_void_p), ("mode", ctypes.c_uint), ("grid_cap", ctypes.c_uint),
                ("block", ctypes.c_uint), ("flags", ctypes.c_uint), ("wake_fn", ctypes.c_void_p), ("stream", ctypes.c_void_p)]


class NVML:
    def __init__(self):
        self.ok = False
        try:
            self.lib = ctypes.CDLL("libnvidia-ml.so.1")
            assert self.lib.nvmlInit_v2() == 0
            self.h = ctypes.c_void_p()
            props = torch.cuda.get_device_properties(0)
            busid = f"{getattr(props, 'pci_domain_id', 0):08x}:{props.pci_bus_id:02x}:{props.pci_device_id:02x}.0".encode()
            if self.lib.nvmlDeviceGetHandleByPciBusId_v2(busid, ctypes.byref(self.h)) != 0:
                assert self.lib.nvmlDeviceGetHandleByIndex_v2(0, ctypes.byref(self.h)) == 0
            self.ok = True
        except Exception:  # noqa: BLE001
            pass

    def sm(self):
        if not self.ok:
            return -1
        c = ctypes.c_uint()
        self.lib.nvmlDeviceGetClockInfo(self.h, 1, ctypes.byref(c))
        return int(c.value)


def load_solution(d, name):
    spec = importlib.util.spec_from_file_location(name, str(d / "solution.py"))
    m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m)
    return m


def load_ext(path, name):
    loader = importlib.machinery.ExtensionFileLoader(name, str(path))
    spec = importlib.util.spec_from_file_location(name, str(path), loader=loader)
    m = importlib.util.module_from_spec(spec)
    loader.exec_module(m)
    return m


class Null(LatDitGenerator):
    def __init__(self, dev):
        self.proto = torch.empty(0, device=dev)

    @classmethod
    def from_pretrained(cls, *a, **k):
        raise NotImplementedError

    def generate_impl(self, vox_idxs, *, conditioning=None, sampling=None, generator=None, **kw):
        return [self.proto.new_empty((vox_idxs[0].shape[0], 16))]


class Holder(LatDitGenerator):
    def __init__(self, fn):
        self.generate_impl = fn

    @classmethod
    def from_pretrained(cls, *a, **k):
        raise NotImplementedError

    def generate_impl(self, vox_idxs, **kw):  # replaced on the instance
        raise NotImplementedError


def main(tag="kernel"):
    dev = torch.device("cuda:0")
    torch.cuda.set_device(dev)
    S6 = load_solution(WORK / "sub6", "solution_v6")
    S7 = load_solution(WORK / "sub7", "solution_v7")
    K8 = load_ext(WORK / "x9" / "k3d_fast9_x86_64.abi3.so", "k3d_fast9")
    nv = NVML()
    files = sorted(VOXDIR.glob("*/cache/vox_idxi.npz"))
    vox = sorted(((p.parent.parent.name, np.load(p)["vox_idxi"]) for p in files), key=lambda kv: len(kv[1]))
    VNP = {i: v for i, v in vox}
    DV = {i: torch.tensor(v, dtype=torch.int32, device=dev) for i, v in vox}
    ids = list(DV)
    sub = [ids[k] for k in sorted(set(np.linspace(0, len(ids) - 1, 8).round().astype(int).tolist()))]
    feats_np = np.random.default_rng(0).standard_normal((1, 4101, 1280)).astype(np.float16)
    tok_np = np.random.default_rng(1).standard_normal((1, 1280)).astype(np.float16)
    cond0 = LatDitConditioning(dino_features=torch.zeros(1, 8, 1280, device=dev), dino_token=torch.zeros(1, 1280, device=dev),
                               batch_size=1)
    autocast = dict(device_type="cuda", dtype=torch.bfloat16 if torch.cuda.get_device_capability(0) >= (8, 0) else torch.float16)
    ba = torch.randn(4096, 4096, device=dev, dtype=torch.float16)
    bc = torch.empty_like(ba)
    rnd = np.random.default_rng(2).random(1 << 20)
    blob = np.random.default_rng(3).integers(0, 255, 4 << 20, dtype=np.uint8).tobytes()

    # rdtsc rate
    t0, c0 = time.perf_counter(), K8.tsc()
    time.sleep(0.5)
    ghz = (K8.tsc() - c0) / (time.perf_counter() - t0) / 1e9
    rep = {"tag": tag, "gpu": torch.cuda.get_device_name(0), "src": {"v6": S6.SRC_TAG, "v7": S7.SRC_TAG}, "tsc_ghz": ghz,
           "idle_s": IDLE_S, "reps": REPS, "objects": {i: int(DV[i].shape[0]) for i in ids}, "subset": sub, "info": {},
           "vs_ref": {}, "fast_stats": {}}
    print(f"[{tag}] {rep['gpu']} | rdtsc {ghz:.3f} GHz | NVML {nv.ok}", flush=True)

    def burst(seconds=0.3):
        t_end = time.perf_counter() + seconds
        while time.perf_counter() < t_end:
            for _ in range(10):
                torch.mm(ba, ba, out=bc)
            torch.cuda.synchronize(dev)

    def cpu_busy(seconds):
        t_end = time.perf_counter() + seconds
        while time.perf_counter() < t_end:
            np.sort(rnd)
            zlib.compress(blob[: 1 << 20], 6)

    def solution(mod, wdir, f, **env):
        old = {k: os.environ.get(k) for k in env}
        os.environ.update(env)
        try:
            return mod.Solution.from_pretrained(str(wdir / f), device=dev)
        finally:
            for k, v in old.items():
                os.environ.pop(k, None) if v is None else os.environ.__setitem__(k, v)

    sol16 = solution(S7, WORK / "sub7" / "weights", "student_n16.pt")
    sol64_v6 = solution(S6, WORK / "sub6" / "weights", "student_n64.pt")
    # the v9 submission zip exactly as packed
    S9 = load_solution(WORK / "sub9", "solution_sub9")
    sol9 = S9.Solution.from_pretrained(str(WORK / "sub9" / "weights"), device=dev)
    rep["info"]["sub9"] = {k: v for k, v in sol9.info.items() if not k.startswith("check")}
    rep["info"]["sub9"]["checks"] = {k: v for k, v in sol9.info.items() if k.startswith("check")}
    print(f"[{tag}] v9 zip: path {sol9.info.get('path')} flags {sol9.info.get('engine_flags', sol9.engine.info.get('flags') if sol9.engine else None)} "
          f"checks {rep['info']['sub9']['checks']}", flush=True)
    eng = sol16.engine
    assert eng is not None, sol16.info
    rep["info"]["v7/n16"] = {k: v for k, v in sol16.info.items() if not k.startswith("check")}
    rep["info"]["v6/n64"] = {k: v for k, v in sol64_v6.info.items() if not k.startswith("check")}
    noop = eng.cu.function(eng.mod, "k7_noop")
    fakes = K8.fakes()
    stream = ctypes.c_void_p()
    eng.cu.check(eng.cu.lib.cuStreamCreate(ctypes.byref(stream), ctypes.c_uint(1)), "cuStreamCreate")  # non-blocking
    keep = [noop, stream]
    dummy_p = S7._Params()
    dummy_pp = (ctypes.c_void_p * 1)(ctypes.addressof(dummy_p))
    keep += [dummy_p, dummy_pp]
    MODES = {"split": 0, "coop": 1, "fused": 2}

    pools = []

    def fast8(mode, flags, real=True, fake=False, on_stream=False):
        pl = Plan8()
        pl.launch, pl.coop = (fakes if fake else (eng.plan.launch, eng.plan.coop))
        if real:
            P, PP = eng.p, eng.pp
            pl.fn[0], pl.fn[1], pl.fn[2] = eng.f_stamp.value, eng.f_mlp.value, eng.f_fused.value
            pl.grid_cap, pl.block = eng.grid_cap, S7.CTA
        else:  # empty kernel everywhere, own parameter block (its parity flips must not touch the engine's)
            P, PP = dummy_p, dummy_pp
            pl.fn[0] = pl.fn[1] = pl.fn[2] = noop.value
            pl.grid_cap, pl.block = 1, 32
        pl.params = ctypes.addressof(PP)
        pl.n = ctypes.addressof(P) + S7._Params.n.offset
        pl.mode, pl.flags = MODES[mode], flags | F_TRACE
        pl.wake_fn = noop.value
        pl.stream = stream.value if on_stream else None
        fn = K8.make(ctypes.addressof(pl), ctypes.addressof(P), eng._proto, torch.int32, sol16._generate_python, 1)
        keep.append(pl)
        if flags & F_DLPACK:  # own slot pool: 4 x 128k voxels
            sb = (1 << 17) * 64
            buf = torch.empty(4 * sb // 4, dtype=torch.float32, device=dev)
            K8.dlpack(fn, torch._C._from_dlpack, buf.data_ptr(), sb, 4, dev.index)
            pools.append(buf)
        return fn

    DLF = F_DLPACK | F_LAUNCH_FIRST
    F = {  # label -> (generator, fast9 callable or None)
        "null": (Null(dev), None),
        "c_only": None, "c_only/dl": None, "noop": None, "noop/dl_first": None,
        "n16": None, "n16/dl": None, "n16/dl_first": None,
        "v7/n16 prod": (sol16, None), "v6/n64 prod": (sol64_v6, None), "v9 prod": (sol9, None),
    }
    for label, args in (("c_only", ("fused", 0, False, True)), ("c_only/dl", ("fused", F_DLPACK, False, True)),
                        ("noop", ("fused", 0, False)), ("noop/dl_first", ("fused", DLF, False)),
                        ("n16", ("fused", 0)), ("n16/dl", ("fused", F_DLPACK)), ("n16/dl_first", ("fused", DLF))):
        fn = fast8(*args)
        F[label] = (Holder(fn), fn)
    eng.reset_state()
    for label, (gen, fn) in F.items():  # correctness of every real variant on every object
        if label in ("null", "c_only", "c_only/dl", "noop", "noop/dl_first"):
            continue
        cs = []
        for v in DV.values():
            out = gen.generate_impl([v], conditioning=None, sampling=None, generator=None)[0]
            torch.cuda.synchronize(dev)
            ref = (sol64_v6 if label.startswith("v6") else sol16).ref.generate(v)  # the v9 zip carries the same n16 weights
            cs.append(S7.compare(out, ref))
        rep["vs_ref"][label] = max(cs, key=lambda c: c["rel_mse"])
        print(f"[{tag}] {label:16s} vs reference: {rep['vs_ref'][label]}", flush=True)
    del out
    import gc
    gc.collect()
    rep["slots_after_check"] = {lb: K8.slots(fn) for lb, (gen, fn) in F.items() if fn is not None}
    if sol9.fast is not None:
        rep["slots_after_check"]["v9 prod"] = sol9.engine._fast_mod.slots(sol9.fast)
    print(f"[{tag}] slots (busy, slot calls) after the checks: {rep['slots_after_check']}", flush=True)

    def call(gen, v, cond=cond0):
        g = torch.Generator(dev).manual_seed(42)
        with torch.no_grad(), torch.autocast(**autocast):
            a = K8.tsc()
            gen.generate([v], conditioning=cond, generator=g)
            b = K8.tsc()
        return gen.last_run_metadata["elapsed_seconds"] * 1e6, a, b

    times, traces = {}, {}

    def ring(fn):
        raw, n = K8.trace(fn)
        return np.frombuffer(raw, np.uint64).reshape(4096, 8).copy(), int(n)

    def collect(label, fn, n0, calls):
        """stage durations (us) of the traced calls n0.. of fn, with the harness stamps around each call"""
        if fn is None:
            return
        rg, n1 = ring(fn)
        recs = [rg[k % 4096] for k in range(n0, n1)]
        if len(recs) != len(calls):  # a call fell back to the Python path (not traced) - keep the totals only
            traces.setdefault(label, []).append({"mismatch": [len(recs), len(calls)]})
            return
        for r, (us, a, b) in zip(recs, calls):
            st = (r[:7].astype(np.int64) - int(r[0])) / (ghz * 1e3)
            traces.setdefault(label, []).append({"us": round(us, 2), "n": int(r[7]),
                                                 "pre": round((int(r[0]) - a) / (ghz * 1e3), 2),
                                                 "stages": [round(float(x), 2) for x in st[1:]],
                                                 "post": round((b - int(r[6])) / (ghz * 1e3), 2)})

    # ---------------------------------------------------------------- warm
    for label, (gen, fn) in F.items():
        n0 = ring(fn)[1] if fn is not None else 0
        res, calls = {}, []
        for i in sub:
            v = DV[i]
            ts = []
            for k in range(23):
                torch.cuda.synchronize(dev)
                c = call(gen, v)
                if k >= 3:
                    ts.append(c[0])
                calls.append(c)
            res[i] = {"n": int(v.shape[0]), "med": float(np.median(ts))}
        times[f"{label}/warm"] = res
        collect(label + "/warm", fn, n0, calls)
        print(f"[{tag}] warm  {label:16s} {np.median([r['med'] for r in res.values()]):6.1f} us", flush=True)

    # ---------------------------------------------------------------- loop / idle, variants interleaved
    def harness_call(gen, i):
        clk = nv.sm()
        v = torch.tensor(VNP[i], dtype=torch.int32, device=dev)
        feats = torch.tensor(feats_np, device=dev)
        tok = torch.tensor(tok_np, device=dev)
        cond = LatDitConditioning(dino_features=feats, dino_token=tok, batch_size=1)
        return call(gen, v, cond) + (clk,)

    for name, prep in (("loop", lambda: (burst(0.3), cpu_busy(IDLE_S))), ("idle", lambda: time.sleep(IDLE_S))):
        n0 = {lb: (ring(fn)[1] if fn is not None else 0) for lb, (gen, fn) in F.items()}
        raw = {lb: {i: [] for i in sub} for lb in F}
        calls = {lb: [] for lb in F}
        for r in range(REPS):
            for i in sub:
                for lb, (gen, fn) in F.items():
                    prep()
                    c = harness_call(gen, i)
                    raw[lb][i].append((round(c[0], 2), c[3]))
                    calls[lb].append(c[:3])
        for lb, (gen, fn) in F.items():
            times[f"{lb}/{name}"] = {i: {"n": int(DV[i].shape[0]), "med": float(np.median([x[0] for x in xs])), "calls": xs}
                                     for i, xs in raw[lb].items()}
            collect(f"{lb}/{name}", fn, n0[lb], calls[lb])
            med = np.median([r["med"] for r in times[f"{lb}/{name}"].values()])
            tr = [t for t in traces.get(f"{lb}/{name}", []) if "stages" in t]
            extra = ""
            if tr:
                pre = np.median([t["pre"] for t in tr])
                st = np.median(np.array([t["stages"] for t in tr]), 0)
                post = np.median([t["post"] for t in tr])
                extra = f" | pre {pre:5.1f} | " + " ".join(f"{s}@{x:5.1f}" for s, x in zip(STAGES, st)) + f" | post {post:5.1f}"
            print(f"[{tag}] {name:5s} {lb:16s} {med:6.1f} us{extra}", flush=True)

    for label, (gen, fn) in F.items():
        if fn is not None:
            rep["fast_stats"][label] = K8.stats(fn)
        elif getattr(gen, "fast", None) is not None:
            rep["fast_stats"][label] = gen.engine._fast_mod.stats(gen.fast)
    rep["times_us"] = times
    rep["traces"] = traces
    rep["slots_end"] = {lb: K8.slots(fn) for lb, (gen, fn) in F.items() if fn is not None}
    if sol9.fast is not None:
        rep["slots_end"]["v9 prod"] = sol9.engine._fast_mod.slots(sol9.fast)
    print(f"[{tag}] slots (busy, slot calls) at the end: {rep['slots_end']}", flush=True)
    out = WORK / "bench" / f"bench11_{tag}.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(rep, indent=1, default=str))
    print(f"[{tag}] fast path calls / fallbacks: {rep['fast_stats']}", flush=True)
    print(f"[{tag}] -> {out}", flush=True)
    return rep


if __name__ == "__main__":
    main(sys.argv[1] if len(sys.argv) > 1 else "subprocess")


In [ ]:
# 1. Веса, cubin, C-шим, C-расширение быстрого пути (base64 zip)
import base64, hashlib
_B64 = (
    "..."
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_B64))) as z:
    for m in z.infolist():
        d, name = m.filename.split('/')
        if d == 'zips':  # the v8 submission zips, unpacked exactly as the platform would
            (WORK / name).write_bytes(z.read(m))
            with zipfile.ZipFile(WORK / name) as zz:
                zz.extractall(WORK / name[:-4])
            continue
        (WORK / d / ('' if d == 'x9' else 'weights') / name).write_bytes(z.read(m))
for d in ('sub6', 'sub7'):
    for f in sorted((WORK / d / 'weights').iterdir()):
        print(f'{d}/{f.name:36s} {f.stat().st_size:>8d}  {hashlib.md5(f.read_bytes()).hexdigest()[:10]}')
print('x9/k3d_fast9_x86_64.abi3.so', hashlib.md5((WORK / 'x9' / 'k3d_fast9_x86_64.abi3.so').read_bytes()).hexdigest()[:10])
for f in ('sub6/solution.py', 'sub7/solution.py', 'sub9/solution.py', 'sub9/weights/k3d_fast9_x86_64.abi3.so', 'bench11.py'):
    print(f, hashlib.md5((WORK / f).read_bytes()).hexdigest()[:10])


In [ ]:
# 2. Реальные воксели: ~24 объекта public_test по всему диапазону N (только vox_idxi.npz, ~10 МБ)
sys.path.insert(0, str(WORK / "k3d" / "calib"))
import k3d_data as D
pat = re.compile(r"(?:^|.*/)([0-9a-f]{32})/cache/vox_idxi\.npz$")
sizes = {}
for m in D.list_remote_zip(D.URLS["public_test"]):
    mm = pat.match(m.filename)
    if mm:
        sizes[mm.group(1)] = (m.file_size - 270) // 12
ids = sorted(sizes, key=lambda i: sizes[i])
q = np.linspace(0, len(ids) - 1, 20).round().astype(int).tolist() + list(range(len(ids) - 4, len(ids)))
pick = sorted({ids[k] for k in q}, key=lambda i: sizes[i])
D.extract_zip_any(D.URLS["public_test"], Path("/tmp/k3d_vox"), lambda n: (pat.match(n) is not None and pat.match(n).group(1) in pick),
                  rename=lambda n: n[n.index(pat.match(n).group(1)):])
print(len(list(Path("/tmp/k3d_vox").glob("*/cache/vox_idxi.npz"))), "объектов")


In [ ]:
# 3. Бенчмарк (~13 мин): сверка, подряд, loop, idle; метки этапов
sys.path.insert(0, str(WORK))
import bench11
REP = bench11.main("kernel")


In [ ]:
# 4. Сводка и архив
def med(k):
    t = REP["times_us"].get(k)
    return float(np.median([r["med"] for r in t.values()])) if t else float("nan")
labels = [k[:-5] for k in REP["times_us"] if k.endswith("/warm")]
print(f"{'вариант':18s} {'warm':>7s} {'loop':>7s} {'idle':>7s}   us, медиана по объектам")
for lb in labels:
    print(f"{lb:18s} {med(lb + '/warm'):7.1f} {med(lb + '/loop'):7.1f} {med(lb + '/idle'):7.1f}")
print()
print("этапы (мкс от входа в C, медианы): до входа | wake checks vox out(+launch при dl_first) launch exit | после выхода")
for k, tr in REP["traces"].items():
    tr = [t for t in tr if "stages" in t]
    if not tr:
        continue
    st = np.median(np.array([t["stages"] for t in tr]), 0)
    print(f"  {k:24s} {np.median([t['pre'] for t in tr]):6.1f} | " + " ".join(f"{x:6.1f}" for x in st) +
          f" | {np.median([t['post'] for t in tr]):6.1f}")
print()
print("быстрый путь (вызовы, откаты):", REP["fast_stats"])
print("сверка с эталоном:", {k: f"{v['rel_mse']:.1e}" for k, v in REP["vs_ref"].items()})
print("слоты (заняты, вызовов через слоты) в конце:", REP.get("slots_end"))
shutil.make_archive(str(WORK / "k3d11_bench"), "zip", root_dir=WORK, base_dir="bench")
print("архив:", WORK / "k3d11_bench.zip")
try:  # Colab: hand the archive to the browser
    from google.colab import files
    files.download(str(WORK / "k3d11_bench.zip"))
except ImportError:
    pass
